# Final Implementation Scope
Η παρούσα υλοποίηση επικεντρώνεται στην ανάπτυξη ενός model-agnostic pipeline για hallucination detection στο πλαίσιο του SHROOM dataset. Το σύστημα αντιμετωπίζει το πρόβλημα ως δυαδική ταξινόμηση, προβλέποντας αν ένα παραγόμενο κείμενο αποτελεί Hallucination ή Not Hallucination, ενώ παράλληλα παράγει και πιθανότητα hallucination.

Η τελική υλοποίηση βασίζεται στα εξής βασικά components:

1. Exploratory Data Analysis (EDA), για την κατανόηση της δομής των δεδομένων, της κατανομής των labels, της κατανομής της p(Hallucination) και των διαφορών μεταξύ των tasks.
2. NLI-based feature extraction, όπου χρησιμοποιούνται entailment, neutral και contradiction scores για τη σύγκριση του hypothesis με το source ή/και το target.
3. Semantic similarity feature extraction, όπου υπολογίζονται similarity scores μεταξύ source, target και hypothesis.
4. Ensemble / meta-learning προσέγγιση, όπου τα παραπάνω χαρακτηριστικά συνδυάζονται με Random Forest μοντέλο για την τελική πρόβλεψη.

Συμπληρωματικά, εξετάζεται η χρήση synthetic data / weak supervision, με στόχο να αξιολογηθεί αν η προσθήκη συνθετικών παραδειγμάτων βελτιώνει την απόδοση του συστήματος. Η επίδραση κάθε component θα αξιολογηθεί μέσω ablation study.

Η υλοποίηση περιορίζεται στο model-agnostic setup. Model-aware τεχνικές, πλήρης RAG υλοποίηση και fine-tuning μεγάλων LLMs δεν αποτελούν μέρος του βασικού πειραματικού συστήματος, αλλά μπορούν να αναφερθούν ως πιθανές επεκτάσεις ή μελλοντική εργασία.


## Components included in the final implementation

- EDA
- NLI feature extraction
- Semantic similarity feature extraction
- Synthetic data generation and filtering
- Ensemble model using Random Forest
- Ablation study
- Error analysis

## Components not included in the final implementation

- Full model-aware setup
- Full RAG pipeline
- Fine-tuning of large LLMs
- Large-scale API-based experimentation

# Experimental Setup and Ablation Study
Σε αυτό το στάδιο οργανώνεται η πειραματική αξιολόγηση του συστήματος. Στόχος δεν είναι μόνο η μέτρηση της τελικής απόδοσης, αλλά και η κατανόηση της συνεισφοράς κάθε επιμέρους component της μεθοδολογίας.

Για τον λόγο αυτό, τα πειράματα οργανώνονται ως ablation study. Ξεκινάμε από απλούστερα baselines και προσθέτουμε σταδιακά επιπλέον χαρακτηριστικά ή δεδομένα. Με αυτόν τον τρόπο μπορούμε να εξετάσουμε αν κάθε τμήμα της υλοποίησης, όπως τα NLI features, τα semantic similarity features και τα synthetic data, προσφέρει πραγματική βελτίωση στην τελική απόδοση.

Η αξιολόγηση πραγματοποιείται με δύο βασικές μετρικές:

* Accuracy, για τη δυαδική ταξινόμηση Hallucination / Not Hallucination.
* Spearman correlation, για τη σύγκριση των προβλεπόμενων πιθανοτήτων hallucination με τις ανθρώπινες εκτιμήσεις p(Hallucination).

Όλα τα πειράματα εκτελούνται με το ίδιο evaluation protocol, ώστε τα αποτελέσματα να είναι συγκρίσιμα μεταξύ τους.


## Final Development Results Summary

| Experiment ID | Experiment | Overall OOF Accuracy | Overall OOF Spearman ρ | Selection status | Main finding |
|---|---|---:|---:|---|---|
| E1 | NLI only | 0.7796 | 0.6905 | Eligible | Τα NLI features αποτελούν ισχυρό baseline |
| E2 | Semantic Similarity only | 0.7054 | 0.5613 | Eligible | Η similarity από μόνη της έχει χαμηλότερη απόδοση |
| E3 | NLI + Similarity | **0.8076** | **0.7388** | **Selected** | Ο συνδυασμός των δύο ομάδων χαρακτηριστικών δίνει την καλύτερη συνολική απόδοση |
| E4 | NLI + Similarity + Synthetic Data | 0.7996 | 0.7186 | Diagnostic only | Τα synthetic δεδομένα δεν βελτιώνουν συνολικά το E3 |

Η τελική επιλογή είναι το E3, καθώς παρουσιάζει την υψηλότερη Overall OOF Accuracy και την υψηλότερη Overall OOF Spearman συσχέτιση μεταξύ των πειραμάτων που συμμετέχουν στην επιλογή μοντέλου. Το E4 αντιμετωπίζεται ως διαγνωστικό ablation experiment και δεν χρησιμοποιείται για την επιλογή της τελικής διαμόρφωσης.

Όλα τα αποτελέσματα προέρχονται από εσωτερική 5-fold cross-validation στο model-agnostic development set των 499 δειγμάτων. Δεν αναφέρονται αποτελέσματα από το official test set ούτε γίνεται σύγκριση με την επίσημη κατάταξη του shared task.

# 1 load

In [ ]:
#imports
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Global configuration
# These constants are used across the whole notebook to ensure reproducibility
# and consistent evaluation across all experiments.

RANDOM_STATE = 42
N_SPLITS = 5
THRESHOLD = 0.5

np.random.seed(RANDOM_STATE)

# Main evaluation dataset for the current implementation.
# The implementation focuses on the model-agnostic setup.
EVALUATION_TRACK = "model-agnostic"

print("Global configuration")
print("RANDOM_STATE:", RANDOM_STATE)
print("N_SPLITS:", N_SPLITS)
print("THRESHOLD:", THRESHOLD)
print("EVALUATION_TRACK:", EVALUATION_TRACK)

from sklearn.metrics import accuracy_score
from scipy.stats import spearmanr
import numpy as np

def evaluate_predictions(y_true_bin, y_true_prob, y_pred_prob, threshold=THRESHOLD):
    """
    Evaluates hallucination detection predictions.

    Parameters
    ----------
    y_true_bin : array-like
        Binary ground-truth labels: 1 for Hallucination, 0 for Not Hallucination.
    y_true_prob : array-like
        Human annotation probability p(Hallucination).
    y_pred_prob : array-like
        Predicted hallucination probability.
    threshold : float
        Threshold used to convert predicted probabilities into binary labels.

    Returns
    -------
    dict
        Accuracy and Spearman correlation.
    """

    y_pred_prob = np.asarray(y_pred_prob)
    y_true_bin = np.asarray(y_true_bin)
    y_true_prob = np.asarray(y_true_prob)

    y_pred_bin = (y_pred_prob >= threshold).astype(int)

    accuracy = accuracy_score(y_true_bin, y_pred_bin)
    spearman = spearmanr(y_true_prob, y_pred_prob).correlation

    return {
        "accuracy": accuracy,
        "spearman": spearman
    }

#paths
#DATA_DIR = Path("../data")   #αν το notebook είναι μέσα σε notebooks/
DATA_DIR = Path((__import__('os').environ.get('SHROOM_PROJECT_ROOT', '.') + '/data'))  #αν το notebook είναι στο root

#unlabeled train
train_agnostic_path = DATA_DIR / "Unlabeled train" / "SHROOM_unlabeled-training-data-v2" / "train.model-agnostic.json"
train_aware_path = DATA_DIR / "Unlabeled train" / "SHROOM_unlabeled-training-data-v2" / "train.model-aware.v2.json"
#trial
trial_path = DATA_DIR / "Trial" / "SHROOM_trial-v1.1" / "trial-v1.json"
#labeled test
test_agnostic_path = DATA_DIR / "Labeled test" / "SHROOM_test-labeled" / "test.model-agnostic.json"
test_aware_path = DATA_DIR / "Labeled test" / "SHROOM_test-labeled" / "test.model-aware.json"

#Labeled development sets
dev_agnostic_path = DATA_DIR / "Dev" / "SHROOM_dev-v2" / "val.model-agnostic.json"


dev_aware_path = DATA_DIR / "Dev" / "SHROOM_dev-v2" / "val.model-aware.v2.json"

In [ ]:
#check paths
print(train_agnostic_path)
print(train_aware_path)
print(trial_path)
print(test_agnostic_path)
print(test_aware_path)
print(dev_agnostic_path)
print(dev_aware_path)

In [ ]:
#function to load json data into a pandas dataframe
def load_json_to_df(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    return df

#data frames
train_agnostic_df = load_json_to_df(train_agnostic_path)
train_aware_df = load_json_to_df(train_aware_path)
trial_df = load_json_to_df(trial_path)
test_agnostic_df = load_json_to_df(test_agnostic_path)
test_aware_df = load_json_to_df(test_aware_path)
dev_agnostic_df = load_json_to_df(dev_agnostic_path)
dev_aware_df = load_json_to_df(dev_aware_path)

#print shapes of dataframes
print("Train agnostic shape:", train_agnostic_df.shape)
print("Train aware shape:", train_aware_df.shape)
print("Trial shape:", trial_df.shape)
print("Test agnostic shape:", test_agnostic_df.shape)
print("Test aware shape:", test_aware_df.shape)
print("Development agnostic:", dev_agnostic_df.shape)
print("Development aware:", dev_aware_df.shape)

#print heads of dataframes
#display(train_agnostic_df.head(2))
#display(train_aware_df.head(2))
#display(trial_df.head(2))
#display(test_agnostic_df.head(2))
#display(test_aware_df.head(2))
#display(test_agnostic_df.head(2))
#display(dev_agnostic_df.head(2))
#display(dev_aware_df.head(2))

# 2. EDA

### 2.1 EDA setup

In [ ]:
# This cell contains helper imports and functions used for saving EDA plots.

import matplotlib.pyplot as plt
from pathlib import Path

# Main figures folder
FIG_ROOT = Path("figures")


# EDA category folders
UNIVARIATE_DIR = FIG_ROOT / "univariate"
BIVARIATE_DIR = FIG_ROOT / "bivariate"
MULTIVARIATE_DIR = FIG_ROOT / "multivariate"

# Univariate plot folders
LABEL_DIST_DIR = UNIVARIATE_DIR / "label_distribution"
TASK_DIST_DIR = UNIVARIATE_DIR / "task_distribution"
P_HALLUCINATION_DIST_DIR = UNIVARIATE_DIR / "p_hallucination_distribution"
TOKEN_LENGTHS_DIR = UNIVARIATE_DIR / "token_lengths"

# Bivariate plot folders
LABEL_BY_TASK_DIR = BIVARIATE_DIR / "label_by_task"
P_HALLUCINATION_BY_TASK_DIR = BIVARIATE_DIR / "p_hallucination_by_task"
LABEL_BY_TRACK_DIR = BIVARIATE_DIR / "label_by_track"

# Multivariate plot folders
LABEL_BY_TASK_AND_TRACK_DIR = MULTIVARIATE_DIR / "label_by_task_and_track"
P_HALLUCINATION_BY_TASK_AND_TRACK_DIR = MULTIVARIATE_DIR / "p_hallucination_by_task_and_track"
TOKEN_LENGTHS_BY_TASK_AND_LABEL_DIR = MULTIVARIATE_DIR / "token_lengths_by_task_and_label"

# Create all folders
for folder in [
    LABEL_DIST_DIR,
    TASK_DIST_DIR,
    P_HALLUCINATION_DIST_DIR,
    TOKEN_LENGTHS_DIR,
    LABEL_BY_TASK_DIR,
    P_HALLUCINATION_BY_TASK_DIR,
    LABEL_BY_TRACK_DIR,
    LABEL_BY_TASK_AND_TRACK_DIR,
    P_HALLUCINATION_BY_TASK_AND_TRACK_DIR,
    TOKEN_LENGTHS_BY_TASK_AND_LABEL_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

def safe_name(name):
    """
    Converts dataset names such as 'Test Agnostic'
    into safe file names such as 'test_agnostic'.
    """
    return name.lower().replace(" ", "_").replace("/", "_")

def save_plot(name, folder):
    """
    Saves the current matplotlib figure as a PNG file
    in the selected folder and then displays it.
    """
    plt.tight_layout()
    plt.savefig(folder / f"{name}.png", dpi=300, bbox_inches="tight")
    plt.show()

### 2.2 Data inspection

In [ ]:
def inspect_dataset(df, name):
    """
    Basic inspection of each dataset:
    - number of rows and columns
    - column names
    - data types
    - missing values
    - duplicate rows
    """

    print("\n" + "="*60)
    print(f"{name} - Basic Inspection")
    print("="*60)

    #πόσες γραμμές και στήλες έχει κάθε dataset
    print("Shape:")
    print(df.shape)

    #ποιες στήλες υπάρχουν
    print("\nColumns:")
    print(df.columns.tolist())

    #τι τύπο δεδομένων έχει κάθε στήλη
    print("\nData types:")
    print(df.dtypes)

    #αν υπάρχουν κενές τιμές
    print("\nMissing values:")
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    if len(missing) == 0:
        print("No missing values.")
    else:
        print(missing)

    print("\nDuplicate rows:")
    
    #αν υπάρχουν διπλότυπες γραμμές
    # Some columns may contain lists, so we convert the dataframe to string only for the duplicate check.
    duplicate_count = df.astype(str).duplicated().sum()
    print(duplicate_count)

In [ ]:
inspect_dataset(trial_df, "Trial")
inspect_dataset(test_agnostic_df, "Test Agnostic")
inspect_dataset(test_aware_df, "Test Aware")

### 2.3 Univariate Analysis

#### 2.3.1 Label distribution

In [ ]:
def label_distribution(df, name):
    """
    Prints the distribution of Hallucination / Not Hallucination labels
    for a given dataset.
    """

    print("\n" + "="*60)
    print(f"{name} - Label Distribution")
    print("="*60)

    counts = df["label"].value_counts(dropna=False)
    percentages = df["label"].value_counts(normalize=True, dropna=False) * 100

    summary = pd.DataFrame({
        "count": counts,
        "percentage": percentages.round(2)
    })

    display(summary)

In [ ]:
label_distribution(trial_df, "Trial")
label_distribution(test_agnostic_df, "Test Agnostic")
label_distribution(test_aware_df, "Test Aware")

#### 2.3.2 Label distribution plots

In [ ]:
def plot_label_distribution(df, name):
    """
    Plots the label distribution for a given dataset.
    """

    counts = df["label"].value_counts(dropna=False)

    plt.figure(figsize=(6, 4))
    ax = counts.plot(kind="bar")

    plt.title(f"{name} - Label Distribution")
    plt.xlabel("Label")
    plt.ylabel("Number of samples")
    plt.xticks(rotation=0)

    for i, value in enumerate(counts.values):
        ax.text(i, value, str(value), ha="center", va="bottom")

    save_plot(f"{safe_name(name)}_label_distribution", folder=LABEL_DIST_DIR)

In [ ]:
# Combined label distribution plot for the two development sets

label_order = ["Hallucination", "Not Hallucination"]

development_label_counts = pd.DataFrame({
    "Model-Agnostic Development":
        dev_agnostic_df["label"].value_counts(),
    "Model-Aware Development":
        dev_aware_df["label"].value_counts()
}).reindex(label_order).fillna(0).astype(int)

ax = development_label_counts.plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Label Distribution in the Development Sets")
plt.xlabel("Label")
plt.ylabel("Number of samples")
plt.xticks(rotation=0)
plt.legend(title="Development Set")

for container in ax.containers:
    ax.bar_label(container)

save_plot(
    "development_label_distribution",
    folder=LABEL_DIST_DIR
)

In [ ]:
plot_label_distribution(trial_df, "Trial")
plot_label_distribution(test_agnostic_df, "Test Agnostic")
plot_label_distribution(test_aware_df, "Test Aware")

#### 2.3.3 Task distribution

In [ ]:
# Combined task distribution plot for the two development sets

task_order = ["DM", "MT", "PG"]

development_task_counts = pd.DataFrame({
    "Model-Agnostic Development":
        dev_agnostic_df["task"].value_counts(),
    "Model-Aware Development":
        dev_aware_df["task"].value_counts()
}).reindex(task_order).fillna(0).astype(int)

display(development_task_counts)

ax = development_task_counts.plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Task Distribution in the Development Sets")
plt.xlabel("Task")
plt.ylabel("Number of samples")
plt.xticks(rotation=0)
plt.legend(title="Development Set")

for container in ax.containers:
    ax.bar_label(container)

save_plot(
    "development_task_distribution",
    folder=TASK_DIST_DIR
)

In [ ]:
def task_distribution(df, name):
    """
    Prints the distribution of NLG tasks for a given dataset.
    """

    print("\n" + "="*60)
    print(f"{name} - Task Distribution")
    print("="*60)

    counts = df["task"].value_counts(dropna=False)
    percentages = df["task"].value_counts(normalize=True, dropna=False) * 100

    summary = pd.DataFrame({
        "count": counts,
        "percentage": percentages.round(2)
    })

    display(summary)

In [ ]:
task_distribution(trial_df, "Trial")
task_distribution(test_agnostic_df, "Test Agnostic")
task_distribution(test_aware_df, "Test Aware")

#### 2.3.4 Task distribution plots

In [ ]:
def plot_task_distribution(df, name):
    """
    Plots the task distribution for a given dataset.
    """

    counts = df["task"].value_counts(dropna=False)

    plt.figure(figsize=(6, 4))
    ax = counts.plot(kind="bar")

    plt.title(f"{name} - Task Distribution")
    plt.xlabel("Task")
    plt.ylabel("Number of samples")
    plt.xticks(rotation=0)

    for i, value in enumerate(counts.values):
        ax.text(i, value, str(value), ha="center", va="bottom")

    save_plot(f"{safe_name(name)}_task_distribution", folder=TASK_DIST_DIR)

In [ ]:
plot_task_distribution(trial_df, "Trial")
plot_task_distribution(test_agnostic_df, "Test Agnostic")
plot_task_distribution(test_aware_df, "Test Aware")

#### 2.3.5 p(Hallucination) distribution

In [ ]:
# Combined p(Hallucination) distribution
# for the two development sets

p_order = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
prob_col = "p(Hallucination)"

development_p_counts = pd.DataFrame({
    "Model-Agnostic Development":
        dev_agnostic_df[prob_col].value_counts(),

    "Model-Aware Development":
        dev_aware_df[prob_col].value_counts()
}).reindex(p_order).fillna(0).astype(int)

display(development_p_counts)

ax = development_p_counts.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title("p(Hallucination) Distribution in the Development Sets")
plt.xlabel("p(Hallucination)")
plt.ylabel("Number of samples")
plt.xticks(
    ticks=range(len(p_order)),
    labels=[f"{value:.1f}" for value in p_order],
    rotation=0
)
plt.legend(title="Development Set")

for container in ax.containers:
    ax.bar_label(container)

save_plot(
    "development_p_hallucination_distribution",
    folder=P_HALLUCINATION_DIST_DIR
)

In [ ]:
def p_hallucination_distribution(df, name, prob_col="p(Hallucination)"):
    """
    Prints the distribution of p(Hallucination) values
    using difficulty bins.
    """

    print("\n" + "="*60)
    print(f"{name} - p(Hallucination) Distribution")
    print("="*60)

    bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    labels = ["0.0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"]

    p_bins = pd.cut(
        df[prob_col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    counts = p_bins.value_counts(sort=False)
    percentages = p_bins.value_counts(normalize=True, sort=False) * 100

    summary = pd.DataFrame({
        "count": counts,
        "percentage": percentages.round(2)
    })

    display(summary)

In [ ]:
p_hallucination_distribution(trial_df, "Trial")
p_hallucination_distribution(test_agnostic_df, "Test Agnostic")
p_hallucination_distribution(test_aware_df, "Test Aware")

#### 2.3.6 p(Hallucination) distribution plots

In [ ]:
def plot_p_hallucination_distribution(df, name, prob_col="p(Hallucination)"):
    """
    Plots the distribution of p(Hallucination) values.
    """

    plt.figure(figsize=(7, 4))

    plt.hist(
        df[prob_col].dropna(),
        bins=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
        edgecolor="black"
    )

    plt.title(f"{name} - p(Hallucination) Distribution")
    plt.xlabel("p(Hallucination)")
    plt.ylabel("Number of samples")

    save_plot(
    f"{safe_name(name)}_p_hallucination_distribution",
    folder=P_HALLUCINATION_DIST_DIR
    )

In [ ]:
plot_p_hallucination_distribution(trial_df, "Trial")
plot_p_hallucination_distribution(test_agnostic_df, "Test Agnostic")
plot_p_hallucination_distribution(test_aware_df, "Test Aware")

#### 2.3.7 Token length feature creation

In [ ]:
from transformers import AutoTokenizer

# We use the tokenizer of the NLI model in order to estimate the actual token lengths that will be used later by transformer-based models.
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")

MAX_LEN = 512

def token_len(text):
    """
    Υπολογίζει το μήκος ενός μεμονωμένου κειμένου σε tokens.
    """
    if pd.isna(text):
        text = ""

    encoded = tokenizer(
        str(text),
        add_special_tokens=True,
        truncation=False
    )

    return len(encoded["input_ids"])


def pair_token_len(text_a, text_b):
    """
    Υπολογίζει το μήκος ενός ζεύγους κειμένων σε tokens.
    """
    if pd.isna(text_a):
        text_a = ""

    if pd.isna(text_b):
        text_b = ""

    encoded = tokenizer(
        str(text_a),
        str(text_b),
        add_special_tokens=True,
        truncation=False
    )

    return len(encoded["input_ids"])

#### 2.3.8 Add token length columns

In [ ]:
from transformers import AutoTokenizer

TOKENIZER_NAME = "microsoft/deberta-v3-large"

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_NAME,
    use_fast=True
)

def token_len(text):
    if pd.isna(text):
        return 0

    return len(
        tokenizer.encode(
            str(text),
            add_special_tokens=False
        )
    )

In [ ]:
def add_token_length_columns(df):
    """
    Adds token length columns for src, hyp, tgt
    and for the pairs src-hyp and tgt-hyp.
    """

    df = df.copy()

    if "src" in df.columns:
        df["src_tokens"] = df["src"].apply(token_len)

    if "hyp" in df.columns:
        df["hyp_tokens"] = df["hyp"].apply(token_len)

    if "tgt" in df.columns:
        df["tgt_tokens"] = df["tgt"].apply(token_len)

    if "src" in df.columns and "hyp" in df.columns:
        df["src_hyp_tokens"] = df.apply(
            lambda x: pair_token_len(x["src"], x["hyp"]),
            axis=1
        )

    if "tgt" in df.columns and "hyp" in df.columns:
        df["tgt_hyp_tokens"] = df.apply(
            lambda x: pair_token_len(x["tgt"], x["hyp"]),
            axis=1
        )

    return df

def pair_token_len(text_a, text_b):
    """
    Υπολογίζει το συνολικό μήκος ενός ζεύγους κειμένων
    όπως ακριβώς εισάγεται στον DeBERTa tokenizer.
    """
    if pd.isna(text_a):
        text_a = ""

    if pd.isna(text_b):
        text_b = ""

    encoded = tokenizer(
        str(text_a),
        str(text_b),
        add_special_tokens=True,
        truncation=False
    )

    return len(encoded["input_ids"])

In [ ]:
trial_df = add_token_length_columns(trial_df)
test_agnostic_df = add_token_length_columns(test_agnostic_df)
test_aware_df = add_token_length_columns(test_aware_df)
dev_agnostic_df = add_token_length_columns(dev_agnostic_df)
dev_aware_df = add_token_length_columns(dev_aware_df)

print("Development token length columns added successfully.")
print(trial_df[["src_tokens", "hyp_tokens", "tgt_tokens", "src_hyp_tokens", "tgt_hyp_tokens"]].head())

In [ ]:
# Token Length Distribution - Development Sets
from pathlib import Path
import matplotlib.pyplot as plt

# Στήλες που δημιουργήθηκαν προηγουμένως
token_cols = [
    "src_tokens",
    "hyp_tokens",
    "tgt_tokens",
    "src_hyp_tokens",
    "tgt_hyp_tokens"
]

display_labels = [
    "src",
    "hyp",
    "tgt",
    "src-hyp",
    "tgt-hyp"
]

# Έλεγχος ότι οι στήλες υπάρχουν και στα δύο development sets
assert set(token_cols).issubset(dev_agnostic_df.columns)
assert set(token_cols).issubset(dev_aware_df.columns)

# Φάκελος αποθήκευσης
TOKEN_LENGTHS_DIR = (
    Path("figures")
    / "univariate"
    / "token_lengths"
)

TOKEN_LENGTHS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Δημιουργία ενιαίου σχήματος με δύο boxplots
fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(14, 5),
    sharey=True
)

# Model-agnostic development
axes[0].boxplot(
    [
        dev_agnostic_df[column].dropna()
        for column in token_cols
    ],
    showfliers=True
)

axes[0].set_title("Model-Agnostic Development")
axes[0].set_xlabel("Text field or input pair")
axes[0].set_ylabel("Number of tokens")
axes[0].set_xticks(
    range(1, len(display_labels) + 1)
)
axes[0].set_xticklabels(
    display_labels,
    rotation=20
)

# Model-aware development
axes[1].boxplot(
    [
        dev_aware_df[column].dropna()
        for column in token_cols
    ],
    showfliers=True
)

axes[1].set_title("Model-Aware Development")
axes[1].set_xlabel("Text field or input pair")
axes[1].set_xticks(
    range(1, len(display_labels) + 1)
)
axes[1].set_xticklabels(
    display_labels,
    rotation=20
)

fig.suptitle(
    "Token Length Distributions in the Development Sets"
)

fig.tight_layout(
    rect=[0, 0, 1, 0.93]
)

# Αποθήκευση
output_path = (
    TOKEN_LENGTHS_DIR
    / "development_token_length_boxplots.png"
)

fig.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)

# Εμφάνιση στο notebook
plt.show()

print("Saved figure to:")
print(output_path.resolve())

In [ ]:
token_cols = [
    "src_tokens",
    "hyp_tokens",
    "tgt_tokens",
    "src_hyp_tokens",
    "tgt_hyp_tokens"
]

print("Model-Agnostic Development")
display(
    dev_agnostic_df[token_cols]
    .mean()
    .round(2)
)

print("Model-Aware Development")
display(
    dev_aware_df[token_cols]
    .mean()
    .round(2)
)

In [ ]:
token_cols = [
    "src_tokens",
    "hyp_tokens",
    "tgt_tokens",
    "src_hyp_tokens",
    "tgt_hyp_tokens"
]

print("Model-Agnostic Development")
display(dev_agnostic_df[token_cols].describe().round(2))

print("Model-Aware Development")
display(dev_aware_df[token_cols].describe().round(2))

#### 2.3.9 Token length distribution

In [ ]:
def token_length_distribution(df, name):
    """
    Prints descriptive statistics for token length columns.
    """

    print("\n" + "="*60)
    print(f"{name} - Token Length Distribution")
    print("="*60)

    cols = [
        "src_tokens",
        "hyp_tokens",
        "tgt_tokens",
        "src_hyp_tokens",
        "tgt_hyp_tokens"
    ]

    available_cols = [col for col in cols if col in df.columns]

    summary = df[available_cols].describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    ).round(2)

    display(summary)

In [ ]:
token_length_distribution(trial_df, "Trial")
token_length_distribution(test_agnostic_df, "Test Agnostic")
token_length_distribution(test_aware_df, "Test Aware")

#### 2.3.10 Token length boxplots

In [ ]:
def plot_token_length_boxplots(df, name):
    """
    Creates one boxplot per dataset for all token length columns.
    This gives a compact overview of the distribution, spread and outliers.
    """

    cols = [
        "src_tokens",
        "hyp_tokens",
        "tgt_tokens",
        "src_hyp_tokens",
        "tgt_hyp_tokens"
    ]

    available_cols = [col for col in cols if col in df.columns]

    if not available_cols:
        print(f"No token length columns found for {name}.")
        return

    data = [df[col].dropna() for col in available_cols]

    plt.figure(figsize=(10, 5))
    plt.boxplot(data, showfliers=True)

    plt.title(f"{name} - Token Length Boxplots")
    plt.xlabel("Token length features")
    plt.ylabel("Number of tokens")
    plt.xticks(
        ticks=range(1, len(available_cols) + 1),
        labels=available_cols,
        rotation=20
    )

    save_plot(
        f"{safe_name(name)}_token_length_boxplots",
        folder=TOKEN_LENGTHS_DIR
    )

In [ ]:
plot_token_length_boxplots(trial_df, "Trial")
plot_token_length_boxplots(test_agnostic_df, "Test Agnostic")
plot_token_length_boxplots(test_aware_df, "Test Aware")

### 2.4 Bivariate Analysis


In [ ]:

# Figure 4.5 - Label distribution by task in the model-agnostic development set

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Έλεγχος ότι υπάρχουν οι απαραίτητες στήλες
required_columns = {"task", "label"}
assert required_columns.issubset(dev_agnostic_df.columns)

# Σειρά εμφάνισης των tasks και των labels
task_order = ["DM", "MT", "PG"]
label_order = ["Hallucination", "Not Hallucination"]

# Απόλυτα πλήθη ανά task και label
label_counts_by_task = pd.crosstab(
    dev_agnostic_df["task"],
    dev_agnostic_df["label"]
)

label_counts_by_task = label_counts_by_task.reindex(
    index=task_order,
    columns=label_order,
    fill_value=0
)

# Ποσοστά μέσα σε κάθε task
label_percentages_by_task = pd.crosstab(
    dev_agnostic_df["task"],
    dev_agnostic_df["label"],
    normalize="index"
).mul(100)

label_percentages_by_task = label_percentages_by_task.reindex(
    index=task_order,
    columns=label_order,
    fill_value=0
)

print("Absolute counts:")
display(label_counts_by_task)

print("Percentages within each task:")
display(label_percentages_by_task.round(2))

# Φάκελος αποθήκευσης
BIVARIATE_DIR = Path("figures") / "bivariate"
BIVARIATE_DIR.mkdir(parents=True, exist_ok=True)

# Δημιουργία grouped bar chart
ax = label_percentages_by_task.plot(
    kind="bar",
    figsize=(9, 5.5),
    width=0.75
)

ax.set_title(
    "Label Distribution by Task\nModel-Agnostic Development Set"
)
ax.set_xlabel("NLG task")
ax.set_ylabel("Percentage of samples (%)")
ax.set_xticklabels(task_order, rotation=0)
ax.set_ylim(0, 100)

ax.legend(
    title="Label",
    loc="upper right"
)

# Εμφάνιση ποσοστών πάνω από τις ράβδους
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        padding=3,
        fontsize=9
    )

plt.tight_layout()

output_path = (
    BIVARIATE_DIR
    / "model_agnostic_label_distribution_by_task.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved figure to:")
print(output_path.resolve())

In [ ]:
# Figure 4.6 - p(Hallucination) distribution by task
# Model-agnostic development set

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

required_columns = {"task", "p(Hallucination)"}
assert required_columns.issubset(dev_agnostic_df.columns)

task_order = ["DM", "MT", "PG"]

# Περιγραφικά στατιστικά ανά task
p_summary_by_task = (
    dev_agnostic_df
    .groupby("task")["p(Hallucination)"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        minimum="min",
        maximum="max"
    )
    .reindex(task_order)
)

print("p(Hallucination) summary by task:")
display(p_summary_by_task.round(3))

# Δεδομένα για τα τρία boxplots
boxplot_data = [
    dev_agnostic_df.loc[
        dev_agnostic_df["task"] == task,
        "p(Hallucination)"
    ].dropna()
    for task in task_order
]

BIVARIATE_DIR = Path("figures") / "bivariate"
BIVARIATE_DIR.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8.5, 5.5))

ax.boxplot(
    boxplot_data,
    labels=task_order,
    showmeans=True
)

ax.set_title(
    "p(Hallucination) Distribution by Task\n"
    "Model-Agnostic Development Set"
)
ax.set_xlabel("NLG task")
ax.set_ylabel("p(Hallucination)")
ax.set_ylim(-0.05, 1.05)
ax.set_yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

output_path = (
    BIVARIATE_DIR
    / "model_agnostic_p_hallucination_by_task.png"
)

fig.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved figure to:")
print(output_path.resolve())

In [ ]:
# Bivariate analysis: token-pair lengths by label
# Model-agnostic development set

import pandas as pd

required_columns = {
    "label",
    "src_hyp_tokens",
    "tgt_hyp_tokens"
}

assert required_columns.issubset(dev_agnostic_df.columns)

label_order = [
    "Hallucination",
    "Not Hallucination"
]

pair_length_columns = [
    "src_hyp_tokens",
    "tgt_hyp_tokens"
]

length_summary_by_label = (
    dev_agnostic_df
    .groupby("label")[pair_length_columns]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max"
    ])
    .reindex(label_order)
    .round(2)
)

print("Token-pair length summary by label:")
display(length_summary_by_label)

In [ ]:
# Figure: Pair-token lengths by label
# Model-agnostic development set

import matplotlib.pyplot as plt
from pathlib import Path

BIVARIATE_DIR = Path("figures") / "bivariate"
BIVARIATE_DIR.mkdir(parents=True, exist_ok=True)

label_order = [
    "Hallucination",
    "Not Hallucination"
]

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(11, 5.5)
)

# src-hyp token lengths
src_hyp_data = [
    dev_agnostic_df.loc[
        dev_agnostic_df["label"] == label,
        "src_hyp_tokens"
    ].dropna()
    for label in label_order
]

axes[0].boxplot(
    src_hyp_data,
    labels=label_order,
    showmeans=True
)

axes[0].set_title("Source–Hypothesis Length by Label")
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Number of tokens")
axes[0].tick_params(axis="x", rotation=15)
axes[0].grid(axis="y", alpha=0.3)

# tgt-hyp token lengths
tgt_hyp_data = [
    dev_agnostic_df.loc[
        dev_agnostic_df["label"] == label,
        "tgt_hyp_tokens"
    ].dropna()
    for label in label_order
]

axes[1].boxplot(
    tgt_hyp_data,
    labels=label_order,
    showmeans=True
)

axes[1].set_title("Target–Hypothesis Length by Label")
axes[1].set_xlabel("Label")
axes[1].set_ylabel("Number of tokens")
axes[1].tick_params(axis="x", rotation=15)
axes[1].grid(axis="y", alpha=0.3)

fig.suptitle(
    "Pair-Token Length Distributions by Label\n"
    "Model-Agnostic Development Set"
)

plt.tight_layout()

output_path = (
    BIVARIATE_DIR
    / "model_agnostic_pair_lengths_by_label.png"
)

fig.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved figure to:")
print(output_path.resolve())

#### 2.4.1 Label distribution by task

In [ ]:
def label_distribution_by_task(df, name):
    """
    Shows how Hallucination / Not Hallucination labels
    are distributed across the different NLG tasks.
    """

    print("\n" + "="*60)
    print(f"{name} - Label Distribution by Task")
    print("="*60)

    counts = pd.crosstab(df["task"], df["label"])
    percentages = pd.crosstab(df["task"], df["label"], normalize="index") * 100

    print("Counts:")
    display(counts)

    print("Percentages per task:")
    display(percentages.round(2))

In [ ]:
label_distribution_by_task(trial_df, "Trial")
label_distribution_by_task(test_agnostic_df, "Test Agnostic")
label_distribution_by_task(test_aware_df, "Test Aware")

#### 2.4.2 Label distribution by task plots

In [ ]:
def plot_label_distribution_by_task(df, name):
    """
    Plots the distribution of labels for each task.
    """

    counts = pd.crosstab(df["task"], df["label"])

    ax = counts.plot(
        kind="bar",
        stacked=True,
        figsize=(7, 4)
    )

    plt.title(f"{name} - Label Distribution by Task")
    plt.xlabel("Task")
    plt.ylabel("Number of samples")
    plt.xticks(rotation=0)
    plt.legend(title="Label")

    save_plot(
        f"{safe_name(name)}_label_by_task",
        folder=LABEL_BY_TASK_DIR
    )

In [ ]:
plot_label_distribution_by_task(trial_df, "Trial")
plot_label_distribution_by_task(test_agnostic_df, "Test Agnostic")
plot_label_distribution_by_task(test_aware_df, "Test Aware")

#### 2.4.3 p(Hallucination) by task

In [ ]:
def p_hallucination_by_task(df, name, prob_col="p(Hallucination)"):
    """
    Shows descriptive statistics of p(Hallucination)
    for each NLG task.
    """

    print("\n" + "="*60)
    print(f"{name} - p(Hallucination) by Task")
    print("="*60)

    summary = (
        df.groupby("task")[prob_col]
        .agg(["count", "mean", "median", "min", "max"])
        .round(3)
    )

    display(summary)

In [ ]:
p_hallucination_by_task(trial_df, "Trial")
p_hallucination_by_task(test_agnostic_df, "Test Agnostic")
p_hallucination_by_task(test_aware_df, "Test Aware")

#### 2.4.4 p(Hallucination) by task plots

In [ ]:
def plot_p_hallucination_by_task(df, name, prob_col="p(Hallucination)"):
    """
    Creates a boxplot of p(Hallucination) by task.
    """

    tasks = sorted(df["task"].dropna().unique())
    data = [df[df["task"] == task][prob_col].dropna() for task in tasks]

    plt.figure(figsize=(7, 4))
    plt.boxplot(data, tick_labels=tasks, showfliers=True)

    plt.title(f"{name} - p(Hallucination) by Task")
    plt.xlabel("Task")
    plt.ylabel("p(Hallucination)")

    save_plot(
        f"{safe_name(name)}_p_hallucination_by_task",
        folder=P_HALLUCINATION_BY_TASK_DIR
    )

In [ ]:
plot_p_hallucination_by_task(trial_df, "Trial")
plot_p_hallucination_by_task(test_agnostic_df, "Test Agnostic")
plot_p_hallucination_by_task(test_aware_df, "Test Aware")

#### 2.4.5 Create track-level dataset

In [ ]:
# Create a combined dataframe for track-level analysis.
# Trial is not included here because the track comparison concerns
# the Test Agnostic and Test Aware datasets.

test_agnostic_track_df = test_agnostic_df.copy()
test_agnostic_track_df["track"] = "model-agnostic"

test_aware_track_df = test_aware_df.copy()
test_aware_track_df["track"] = "model-aware"

test_tracks_df = pd.concat(
    [test_agnostic_track_df, test_aware_track_df],
    ignore_index=True
)

print("Combined track-level dataframe created successfully.")
print(test_tracks_df["track"].value_counts())

#### 2.4.6 Label distribution by track

In [ ]:
def label_distribution_by_track(df):
    """
    Shows how Hallucination / Not Hallucination labels
    are distributed across tracks.
    """

    print("\n" + "="*60)
    print("Label Distribution by Track")
    print("="*60)

    counts = pd.crosstab(df["track"], df["label"])
    percentages = pd.crosstab(df["track"], df["label"], normalize="index") * 100

    print("Counts:")
    display(counts)

    print("Percentages per track:")
    display(percentages.round(2))

In [ ]:
label_distribution_by_track(test_tracks_df)

#### 2.4.7 Label distribution by track plot

In [ ]:
def plot_label_distribution_by_track(df):
    """
    Plots the distribution of labels for each track.
    """

    counts = pd.crosstab(df["track"], df["label"])

    ax = counts.plot(
        kind="bar",
        stacked=True,
        figsize=(7, 4)
    )

    plt.title("Label Distribution by Track")
    plt.xlabel("Track")
    plt.ylabel("Number of samples")
    plt.xticks(rotation=0)
    plt.legend(title="Label")

    save_plot(
        "label_by_track",
        folder=LABEL_BY_TRACK_DIR
    )

In [ ]:
plot_label_distribution_by_track(test_tracks_df)

### 2.5 Limited Multivariate Analysis



In [ ]:
# Figure 4.8 - Hallucination percentage by task and track

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

task_order = ["DM", "MT", "PG"]

def hallucination_percentage_by_task(df, track_name):
    result = (
        df.assign(
            is_hallucination=df["label"].eq("Hallucination")
        )
        .groupby("task")["is_hallucination"]
        .mean()
        .mul(100)
        .reindex(task_order)
    )

    return pd.DataFrame({
        "task": task_order,
        "track": track_name,
        "hallucination_percentage": result.values
    })

agnostic_percentages = hallucination_percentage_by_task(
    dev_agnostic_df,
    "Model-Agnostic"
)

aware_percentages = hallucination_percentage_by_task(
    dev_aware_df,
    "Model-Aware"
)

multivariate_label_df = pd.concat(
    [agnostic_percentages, aware_percentages],
    ignore_index=True
)

display(
    multivariate_label_df.pivot(
        index="task",
        columns="track",
        values="hallucination_percentage"
    ).round(2)
)

MULTIVARIATE_DIR = Path("figures") / "multivariate"
MULTIVARIATE_DIR.mkdir(parents=True, exist_ok=True)

plot_df = multivariate_label_df.pivot(
    index="task",
    columns="track",
    values="hallucination_percentage"
).reindex(task_order)

ax = plot_df.plot(
    kind="bar",
    figsize=(9, 5.5),
    width=0.75
)

ax.set_title(
    "Hallucination Percentage by Task and Track"
)
ax.set_xlabel("NLG task")
ax.set_ylabel("Hallucination samples (%)")
ax.set_xticklabels(task_order, rotation=0)
ax.set_ylim(0, 70)
ax.legend(title="Development set")

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        padding=3,
        fontsize=9
    )

plt.tight_layout()

output_path = (
    MULTIVARIATE_DIR
    / "hallucination_percentage_by_task_and_track.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(output_path.resolve())

In [ ]:
# Figure 4.9 - p(Hallucination) distribution by task and track

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

required_columns = {"task", "p(Hallucination)"}

assert required_columns.issubset(dev_agnostic_df.columns)
assert required_columns.issubset(dev_aware_df.columns)

task_order = ["DM", "MT", "PG"]

# Περιγραφικά στατιστικά για κάθε task και track
agnostic_summary = (
    dev_agnostic_df
    .groupby("task")["p(Hallucination)"]
    .agg(count="count", mean="mean", median="median")
    .reindex(task_order)
)

aware_summary = (
    dev_aware_df
    .groupby("task")["p(Hallucination)"]
    .agg(count="count", mean="mean", median="median")
    .reindex(task_order)
)

summary_table = pd.concat(
    {
        "Model-Agnostic": agnostic_summary,
        "Model-Aware": aware_summary
    },
    axis=1
)

print("p(Hallucination) summary by task and track:")
display(summary_table.round(3))

# Δεδομένα για τα boxplots
agnostic_data = [
    dev_agnostic_df.loc[
        dev_agnostic_df["task"] == task,
        "p(Hallucination)"
    ].dropna()
    for task in task_order
]

aware_data = [
    dev_aware_df.loc[
        dev_aware_df["task"] == task,
        "p(Hallucination)"
    ].dropna()
    for task in task_order
]

MULTIVARIATE_DIR = Path("figures") / "multivariate"
MULTIVARIATE_DIR.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(11, 5.5),
    sharey=True
)

axes[0].boxplot(
    agnostic_data,
    labels=task_order,
    showmeans=True
)

axes[0].set_title("Model-Agnostic Development")
axes[0].set_xlabel("NLG task")
axes[0].set_ylabel("p(Hallucination)")
axes[0].set_ylim(-0.05, 1.05)
axes[0].set_yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
axes[0].grid(axis="y", alpha=0.3)

axes[1].boxplot(
    aware_data,
    labels=task_order,
    showmeans=True
)

axes[1].set_title("Model-Aware Development")
axes[1].set_xlabel("NLG task")
axes[1].set_ylim(-0.05, 1.05)
axes[1].set_yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
axes[1].grid(axis="y", alpha=0.3)

fig.suptitle(
    "p(Hallucination) Distribution by Task and Track"
)

plt.tight_layout()

output_path = (
    MULTIVARIATE_DIR
    / "p_hallucination_by_task_and_track.png"
)

fig.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved figure to:")
print(output_path.resolve())

In [ ]:
# Figure 4.10 - Pair-token lengths by task and label
# Model-agnostic development set

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.patches import Patch

required_columns = {
    "task",
    "label",
    "src_hyp_tokens",
    "tgt_hyp_tokens"
}

assert required_columns.issubset(dev_agnostic_df.columns)

task_order = ["DM", "MT", "PG"]
label_order = ["Hallucination", "Not Hallucination"]

# Περιγραφικά στατιστικά
length_summary_task_label = (
    dev_agnostic_df
    .groupby(["task", "label"])[
        ["src_hyp_tokens", "tgt_hyp_tokens"]
    ]
    .agg(["count", "mean", "median", "std"])
    .reindex(
        pd.MultiIndex.from_product(
            [task_order, label_order],
            names=["task", "label"]
        )
    )
    .round(2)
)

print("Pair-token length summary by task and label:")
display(length_summary_task_label)

MULTIVARIATE_DIR = Path("figures") / "multivariate"
MULTIVARIATE_DIR.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(12, 5.8)
)

def grouped_boxplot(ax, column, title):
    hallucination_data = [
        dev_agnostic_df.loc[
            (dev_agnostic_df["task"] == task)
            & (dev_agnostic_df["label"] == "Hallucination"),
            column
        ].dropna()
        for task in task_order
    ]

    not_hallucination_data = [
        dev_agnostic_df.loc[
            (dev_agnostic_df["task"] == task)
            & (dev_agnostic_df["label"] == "Not Hallucination"),
            column
        ].dropna()
        for task in task_order
    ]

    hallucination_positions = [0.82, 1.82, 2.82]
    not_hallucination_positions = [1.18, 2.18, 3.18]

    hallucination_boxes = ax.boxplot(
        hallucination_data,
        positions=hallucination_positions,
        widths=0.28,
        patch_artist=True,
        showmeans=True
    )

    not_hallucination_boxes = ax.boxplot(
        not_hallucination_data,
        positions=not_hallucination_positions,
        widths=0.28,
        patch_artist=True,
        showmeans=True
    )

    for box in hallucination_boxes["boxes"]:
        box.set_facecolor("white")
        box.set_hatch("//")

    for box in not_hallucination_boxes["boxes"]:
        box.set_facecolor("white")

    ax.set_title(title)
    ax.set_xlabel("NLG task")
    ax.set_ylabel("Number of tokens")
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(task_order)
    ax.grid(axis="y", alpha=0.3)

grouped_boxplot(
    axes[0],
    "src_hyp_tokens",
    "Source–Hypothesis Length"
)

grouped_boxplot(
    axes[1],
    "tgt_hyp_tokens",
    "Target–Hypothesis Length"
)

legend_handles = [
    Patch(
        facecolor="white",
        edgecolor="black",
        hatch="//",
        label="Hallucination"
    ),
    Patch(
        facecolor="white",
        edgecolor="black",
        label="Not Hallucination"
    )
]

fig.legend(
    handles=legend_handles,
    title="Label",
    loc="upper right",
    bbox_to_anchor=(0.97, 0.92)
)

fig.suptitle(
    "Pair-Token Length Distributions by Task and Label\n"
    "Model-Agnostic Development Set"
)

plt.tight_layout(rect=[0, 0, 0.94, 0.92])

output_path = (
    MULTIVARIATE_DIR
    / "pair_lengths_by_task_and_label.png"
)

fig.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved figure to:")
print(output_path.resolve())

#### 2.5.1 Label distribution by task and track

In [ ]:
def label_distribution_by_task_and_track(df):
    """
    Shows how labels are distributed across task and track.
    """

    print("\n" + "="*60)
    print("Label Distribution by Task and Track")
    print("="*60)

    counts = pd.crosstab(
        [df["track"], df["task"]],
        df["label"]
    )

    percentages = pd.crosstab(
        [df["track"], df["task"]],
        df["label"],
        normalize="index"
    ) * 100

    print("Counts:")
    display(counts)

    print("Percentages per track-task group:")
    display(percentages.round(2))

In [ ]:
label_distribution_by_task_and_track(test_tracks_df)

#### 2.5.2 Label distribution by task and track plot

In [ ]:
def plot_label_distribution_by_task_and_track(df):
    """
    Plots label distribution for each task-track combination.
    """

    counts = pd.crosstab(
        [df["track"], df["task"]],
        df["label"]
    )

    ax = counts.plot(
        kind="bar",
        stacked=True,
        figsize=(9, 5)
    )

    plt.title("Label Distribution by Task and Track")
    plt.xlabel("Track / Task")
    plt.ylabel("Number of samples")
    plt.xticks(rotation=30, ha="right")
    plt.legend(title="Label")

    save_plot(
        "label_by_task_and_track",
        folder=LABEL_BY_TASK_AND_TRACK_DIR
    )

In [ ]:
plot_label_distribution_by_task_and_track(test_tracks_df)

#### 2.5.3 p(Hallucination) by task and track

In [ ]:
def p_hallucination_by_task_and_track(df, prob_col="p(Hallucination)"):
    """
    Shows descriptive statistics of p(Hallucination)
    for each task-track combination.
    """

    print("\n" + "="*60)
    print("p(Hallucination) by Task and Track")
    print("="*60)

    summary = (
        df.groupby(["track", "task"])[prob_col]
        .agg(["count", "mean", "median", "min", "max"])
        .round(3)
    )

    display(summary)

In [ ]:
p_hallucination_by_task_and_track(test_tracks_df)

#### 2.5.4 p(Hallucination) by task and track plot

In [ ]:
def plot_p_hallucination_by_task_and_track(df, prob_col="p(Hallucination)"):
    """
    Creates grouped boxplots of p(Hallucination)
    for each task-track combination.
    """

    groups = []
    labels = []

    for track in sorted(df["track"].dropna().unique()):
        for task in sorted(df["task"].dropna().unique()):
            values = df[
                (df["track"] == track) &
                (df["task"] == task)
            ][prob_col].dropna()

            if len(values) > 0:
                groups.append(values)
                labels.append(f"{track}\n{task}")

    plt.figure(figsize=(9, 5))
    plt.boxplot(groups, showfliers=True)

    plt.title("p(Hallucination) by Task and Track")
    plt.xlabel("Track / Task")
    plt.ylabel("p(Hallucination)")
    plt.xticks(
        ticks=range(1, len(labels) + 1),
        labels=labels,
        rotation=0
    )

    save_plot(
        "p_hallucination_by_task_and_track",
        folder=P_HALLUCINATION_BY_TASK_AND_TRACK_DIR
    )

In [ ]:
plot_p_hallucination_by_task_and_track(test_tracks_df)

#### 2.5.5 Token lengths by task and label

In [ ]:
def token_lengths_by_task_and_label(df, name):
    """
    Shows descriptive statistics of token lengths
    grouped by task and label.
    """

    print("\n" + "="*60)
    print(f"{name} - Token Lengths by Task and Label")
    print("="*60)

    cols = [
        "src_tokens",
        "hyp_tokens",
        "tgt_tokens",
        "src_hyp_tokens",
        "tgt_hyp_tokens"
    ]

    available_cols = [col for col in cols if col in df.columns]

    if not available_cols:
        print(f"No token length columns found for {name}.")
        return

    summary = (
        df.groupby(["task", "label"])[available_cols]
        .agg(["count", "mean", "median", "max"])
        .round(2)
    )

    display(summary)

In [ ]:
token_lengths_by_task_and_label(trial_df, "Trial")
token_lengths_by_task_and_label(test_agnostic_df, "Test Agnostic")
token_lengths_by_task_and_label(test_aware_df, "Test Aware")

#### 2.5.6 Token lengths by task and label plots

In [ ]:
def plot_token_lengths_by_task_and_label(df, name):
    """
    Creates boxplots for pair token lengths grouped by task and label.
    We focus on src_hyp_tokens and tgt_hyp_tokens because these are
    the most relevant input pairs for NLI-based analysis.
    """

    pair_cols = ["src_hyp_tokens", "tgt_hyp_tokens"]

    for col in pair_cols:
        if col not in df.columns:
            print(f"{col} not found in {name}. Skipping.")
            continue

        groups = []
        group_labels = []

        tasks = sorted(df["task"].dropna().unique())
        labels = sorted(df["label"].dropna().unique())

        for task in tasks:
            for label in labels:
                values = df[
                    (df["task"] == task) &
                    (df["label"] == label)
                ][col].dropna()

                if len(values) > 0:
                    groups.append(values)
                    group_labels.append(f"{task}\n{label}")

        if not groups:
            print(f"No groups available for {col} in {name}.")
            continue

        plt.figure(figsize=(10, 5))
        plt.boxplot(groups, showfliers=True)

        plt.title(f"{name} - {col} by Task and Label")
        plt.xlabel("Task / Label")
        plt.ylabel("Number of tokens")
        plt.xticks(
            ticks=range(1, len(group_labels) + 1),
            labels=group_labels,
            rotation=30,
            ha="right"
        )

        save_plot(
            f"{safe_name(name)}_{col}_by_task_and_label",
            folder=TOKEN_LENGTHS_BY_TASK_AND_LABEL_DIR
        )

In [ ]:
plot_token_lengths_by_task_and_label(trial_df, "Trial")
plot_token_lengths_by_task_and_label(test_agnostic_df, "Test Agnostic")
plot_token_lengths_by_task_and_label(test_aware_df, "Test Aware")

# 3 NLI BASELINE

In [ ]:
!pip install transformers torch scikit-learn pandas numpy scipy tqdm
%pip install torch transformers

In [ ]:
#imports
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from transformers import pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score
from scipy.stats import spearmanr

In [ ]:
#checking which dataframe has label or labels column
dfs = {
    "train_agnostic_df": train_agnostic_df,
    "train_aware_df": train_aware_df,
    "trial_df": trial_df,
    "dev_agnostic_df": dev_agnostic_df,
    "dev_aware_df": dev_aware_df,
    "test_agnostic_df": test_agnostic_df,
    "test_aware_df": test_aware_df
}

for name, temp_df in dfs.items():
    print("\n" + name)
    print("shape:", temp_df.shape)
    print("columns:", temp_df.columns.tolist())

In [ ]:
#Choosing dataset
#Για τα τελικά πειράματα χρησιμοποιούμε μόνο το model-agnostic development set.
df = dev_agnostic_df.copy().reset_index(drop=True)

label_col = "label"
prob_col = "p(Hallucination)"

print("Selected dataset: model-agnostic development set")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

display(
    df[["task", "src", "hyp", "tgt", label_col, prob_col]].head()
)

assert len(df) == 499, f"Expected 499 development samples, found {len(df)}"

In [ ]:
#Targets
#Making the targets for training and evaluation

#Target for regression
df["p_hallucination"] = df[prob_col]

#Target for binary accuracy
df["label_bin"] = df[label_col].map({
    "Hallucination": 1,
    "Not Hallucination": 0
})

#head
print(df[ [label_col, "label_bin", "p_hallucination"] ].head())
print()
print(df["p_hallucination"].value_counts(dropna=False))
print()
print(df["label_bin"].value_counts(dropna=False))

In [ ]:
#Load NLI model
#NLI model will give us probabilities for entailment, neutral, contradiction which we can use as features in our ensemble 
device = -1  # -1 for CPU και 0 για GPU (αν υπάρχει διαθέσιμη)

nli_model_name = "microsoft/deberta-large-mnli"

nli_pipe = pipeline(
    "text-classification",
    model=nli_model_name,
    tokenizer=nli_model_name,
    truncation=True,
    max_length=512,
    device=device,
    top_k=None   #επιστρέφει όλα τα labels, όχι μόνο το καλύτερο
)

In [ ]:
#Quick NLI check
test_result = nli_pipe({
    "text": "A man is walking in the park.",
    "text_pair": "A person is walking outside."
})

test_result

In [ ]:
#Function to get NLI scores
#returns clear the three NLI scores

#Συνάρτηση που παίρνει δύο προτάσεις:
# premise: η αρχική / βασική πρόταση
# hypothesis: η πρόταση που θέλουμε να συγκρίνουμε με την αρχική
# pipe: το έτοιμο NLI μοντέλο / pipeline, φορτώθηκε πριν δύο cells
#
#Η συνάρτηση επιστρέφει τρία scores:
# entailment, neutral, contradiction

def get_nli_scores(premise, hypothesis, pipe):

    #Στέλνουμε τις δύο προτάσεις στο NLI μοντέλο.
    #Το "text" είναι το premise.
    #Το "text_pair" είναι το hypothesis.
    #Χρησιμοποιούμε str() για να είμαστε σίγουροι ότι είναι κείμενο.
    result = pipe({
        "text": str(premise),
        "text_pair": str(hypothesis)
    })

    #Κάποιες εκδόσεις του transformers επιστρέφουν το αποτέλεσμα
    #μέσα σε διπλή λίστα, δηλαδή:
    # [[ {...}, {...}, {...} ]]
    #
    #Αν συμβεί αυτό, κρατάμε μόνο την εσωτερική λίστα:
    # [ {...}, {...}, {...} ]
    if isinstance(result, list) and len(result) > 0 and isinstance(result[0], list):
        result = result[0]

    #Μετατρέπουμε τη λίστα αποτελεσμάτων σε dictionary.
    #Παράδειγμα:
    #Από:
    # [
    #   {"label": "entailment", "score": 0.80},
    #   {"label": "neutral", "score": 0.15},
    #   {"label": "contradiction", "score": 0.05}
    # ]
    #
    #Γίνεται:
    # {
    #   "entailment": 0.80,
    #   "neutral": 0.15,
    #   "contradiction": 0.05
    # }

    #Το lower() κάνει τα labels πεζά, για να μη μας νοιάζει
    #αν το μοντέλο επιστρέψει "ENTAILMENT" ή "entailment".
    scores = {
        item["label"].lower(): item["score"]
        for item in result
    }

    #Παίρνουμε ξεχωριστά το score για entailment.
    #Αν δεν υπάρχει τέτοιο label, επιστρέφουμε 0.0.
    ent = scores.get("entailment", 0.0)

    #Παίρνουμε ξεχωριστά το score για neutral.
    #Αν δεν υπάρχει τέτοιο label, επιστρέφουμε 0.0.
    neu = scores.get("neutral", 0.0)

    #Παίρνουμε ξεχωριστά το score για contradiction.
    #Αν δεν υπάρχει τέτοιο label, επιστρέφουμε 0.0.
    con = scores.get("contradiction", 0.0)

    #Επιστρέφουμε τα τρία scores με σταθερή σειρά:
    #1. entailment
    #2. neutral
    #3. contradiction
    return ent, neu, con

In [ ]:
#Check score function
ent, neu, con = get_nli_scores(
    "A man is walking in the park.",
    "A person is walking outside.",
    nli_pipe
)

print("Entailment:", ent)
print("Neutral:", neu)
print("Contradiction:", con) 

In [ ]:
#Time estimation
import time

sample_df = df.head(20).copy()

start_time = time.time()

#Για κάθε γραμμή του sample_df, πάρε τη γραμμή ως row, αγνόησε το index (_), και δείξε μπάρα προόδου μέχρι να τελειώσουν όλες οι γραμμές.
for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    #function calls
    get_nli_scores(row["src"], row["hyp"], nli_pipe)
    get_nli_scores(row["tgt"], row["hyp"], nli_pipe)

end_time = time.time()

elapsed_time = end_time - start_time
time_per_sample = elapsed_time / len(sample_df)
estimated_total_time = time_per_sample * len(df)

print(f"Χρόνος για 20 δείγματα: {elapsed_time:.2f} δευτερόλεπτα")
print(f"Μέσος χρόνος ανά δείγμα: {time_per_sample:.2f} δευτερόλεπτα")
print(f"Εκτίμηση για όλο το dataset: {estimated_total_time / 60:.2f} λεπτά")

In [ ]:
#Full NLI feature extraction
#Extracting the 6 NLI features for each sample: src_ent, src_neu, src_con, tgt_ent, tgt_neu, tgt_con

#Λίστες όπου θα αποθηκεύσουμε τα NLI scores για κάθε δείγμα
src_ent_list = []
src_neu_list = []
src_con_list = []

tgt_ent_list = []
tgt_neu_list = []
tgt_con_list = []

#Τρέχουμε NLI σε όλο το dataframe
for _, row in tqdm(df.iterrows(), total=len(df)):

    # Pair 1: source -> hypothesis
    src_ent, src_neu, src_con = get_nli_scores(
        row["src"],
        row["hyp"],
        nli_pipe
    )

    # Pair 2: target -> hypothesis
    tgt_ent, tgt_neu, tgt_con = get_nli_scores(
        row["tgt"],
        row["hyp"],
        nli_pipe
    )

    src_ent_list.append(src_ent)
    src_neu_list.append(src_neu)
    src_con_list.append(src_con)

    tgt_ent_list.append(tgt_ent)
    tgt_neu_list.append(tgt_neu)
    tgt_con_list.append(tgt_con)

#Προσθέτουμε τα 6 NLI features στο dataframe
df["src_ent"] = src_ent_list
df["src_neu"] = src_neu_list
df["src_con"] = src_con_list

df["tgt_ent"] = tgt_ent_list
df["tgt_neu"] = tgt_neu_list
df["tgt_con"] = tgt_con_list

#Έλεγχος ότι δημιουργήθηκαν σωστά
df[[
    "src_ent", "src_neu", "src_con",
    "tgt_ent", "tgt_neu", "tgt_con"
]].head()

In [ ]:
# Saving the development dataset with NLI features
nli_dev_path = "nli_features_dev_agnostic.csv"

df.to_csv(nli_dev_path, index=False)

print("Saved:", nli_dev_path)
print("Shape:", df.shape)

In [ ]:
# Final check for development NLI features

nli_feature_cols = [
    "src_ent", "src_neu", "src_con",
    "tgt_ent", "tgt_neu", "tgt_con"
]

print("Rows:", len(df))
print("Missing NLI values:", df[nli_feature_cols].isna().sum().sum())

print("\nSource score sums:")
print(df[["src_ent", "src_neu", "src_con"]].sum(axis=1).describe())

print("\nTarget score sums:")
print(df[["tgt_ent", "tgt_neu", "tgt_con"]].sum(axis=1).describe())

assert len(df) == 499
assert df[nli_feature_cols].isna().sum().sum() == 0

In [ ]:
# Common folds for E1, E2, E3 and E4

N_SPLITS = 5
RANDOM_STATE = 42
THRESHOLD = 0.5

stratify_key = (
    df["task"].astype(str)
    + "_"
    + df["label_bin"].astype(str)
)

common_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

common_folds = list(
    common_cv.split(
        np.zeros(len(df)),
        stratify_key
    )
)

print("Number of folds:", len(common_folds))

for fold, (train_idx, val_idx) in enumerate(common_folds, start=1):
    print(
        f"Fold {fold}: "
        f"train={len(train_idx)}, validation={len(val_idx)}"
    )

assert len(common_folds) == 5
assert sum(len(val_idx) for _, val_idx in common_folds) == 499

## E1 - NLI-only baseline

In [ ]:
#Φτιάχνουμε το feature matrix και τα targets για το Random Forest baseline.

nli_feature_cols  = [
    "src_ent", "src_neu", "src_con",
    "tgt_ent", "tgt_neu", "tgt_con"
]

#Φτιάχνει το feature matrix με τις 6 NLI στήλες
nli_X = df[nli_feature_cols ].values
#Eίναι το regression target. Δηλαδή κρατάει πιθανότητα hallucination: 0.0, 0.2, 0.4, 0.6, 0.8, 1.0
y_reg = df["p_hallucination"].values
#Eίναι το classification target. Δηλαδή 1 για Hallucination και 0 για Not Hallucination
y_bin = df["label_bin"].values
#Kρατάει το task κάθε δείγματος. Δηλαδή MT, DM, PG
tasks = df["task"].values

print("nli_X shape:", nli_X.shape)
print("y_reg shape:", y_reg.shape)
print("y_bin shape:", y_bin.shape)

In [ ]:
#Eκπαιδεύουμε Random Forest με 5-fold cross-validation πάνω στα NLI features.
fold_results = []

all_val_preds = []
all_val_true_reg = []
all_val_true_bin = []
all_val_tasks = []

#Δημιουργεί ένα σύνθετο stratification key που συνδυάζει task και label.
#Έτσι κάθε fold κρατά περίπου την ίδια κατανομή όχι μόνο σε Hallucination / Not Hallucination,
#αλλά και ανά task (DM, MT, PG), κάτι σημαντικό γιατί τα tasks έχουν διαφορετική δυσκολία.
#Για κάθε fold του cross-validation, πάρε τα training indexes και τα validation indexes και τρέξε το μοντέλο.
for fold, (train_idx, val_idx) in enumerate(common_folds,start=1):
    X_train = nli_X[train_idx]
    X_val = nli_X[val_idx]

    #Χωρίζει τα features σε training και validation set
    y_train_reg = y_reg[train_idx]
    y_val_reg = y_reg[val_idx]
    #Χωρίζει το regression target, δηλαδή το p_hallucination
    y_val_bin = y_bin[val_idx]

    #Κρατάει τα πραγματικά binary labels για το validation set, για να υπολογίσουμε accuracy
    val_tasks = tasks[val_idx]

    #Εκπαιδεύει Random Forest regressor, όχι classifier. Δηλαδή μαθαίνει να προβλέπει: p_hallucination: 0.0, 0.2, 0.4, 0.6, 0.8, 1.0
    model = RandomForestRegressor(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    model.fit(X_train, y_train_reg)

    #Mετατρέπει την προβλεπόμενη πιθανότητα σε binary label
    #>= 0.5  -> Hallucination
    #< 0.5   -> Not Hallucination
    val_pred = model.predict(X_val)
    val_pred = np.clip(val_pred, 0.0, 1.0)

    # Για accuracy μετατρέπουμε την πιθανότητα σε binary label.
    val_pred_bin = (val_pred >= THRESHOLD).astype(int)

    metrics = evaluate_predictions(
        y_true_bin=y_val_bin,
        y_true_prob=y_val_reg,
        y_pred_prob=val_pred,
        threshold=THRESHOLD
    )

    acc = metrics["accuracy"]
    sp = metrics["spearman"]

    fold_results.append({
        "fold": fold,
        "accuracy": acc,
        "spearman": sp
    })

    all_val_preds.extend(val_pred)
    all_val_true_reg.extend(y_val_reg)
    all_val_true_bin.extend(y_val_bin)
    all_val_tasks.extend(val_tasks)

nli_results_df = pd.DataFrame(fold_results)
nli_results_df

In [ ]:
#Yπολογίζουμε συνολικά Accuracy και Spearman για τα out-of-fold predictions.

overall_pred = np.array(all_val_preds)
overall_true_reg = np.array(all_val_true_reg)
overall_true_bin = np.array(all_val_true_bin)

overall_pred_bin = (overall_pred >= THRESHOLD).astype(int)

overall_acc = accuracy_score(overall_true_bin, overall_pred_bin)
overall_sp = spearmanr(overall_true_reg, overall_pred).correlation

print("Mean CV Accuracy:", nli_results_df["accuracy"].mean())
print("Mean CV Spearman:", nli_results_df["spearman"].mean())
print("Overall OOF Accuracy:", overall_acc)
print("Overall OOF Spearman:", overall_sp)

In [ ]:
#Eλέγχουμε την απόδοση ξεχωριστά για κάθε task.

nli_oof_df = pd.DataFrame({
    "task": all_val_tasks,
    "y_true_reg": all_val_true_reg,
    "y_true_bin": all_val_true_bin,
    "y_pred_reg": all_val_preds
})

nli_oof_df["y_pred_bin"] = (nli_oof_df["y_pred_reg"] >= THRESHOLD).astype(int)

for task_name in sorted(nli_oof_df["task"].unique()):

    task_df = nli_oof_df[nli_oof_df["task"] == task_name]

    acc = accuracy_score(
        task_df["y_true_bin"],
        task_df["y_pred_bin"]
    )

    sp = spearmanr(
        task_df["y_true_reg"],
        task_df["y_pred_reg"]
    ).correlation

    print(f"{task_name} -> Accuracy: {acc:.4f}, Spearman: {sp:.4f}")

In [ ]:
nli_oof_df.to_csv(
    "e1_nli_dev_oof_predictions.csv",
    index=False
)

nli_results_df.to_csv(
    "e1_nli_dev_cv_results.csv",
    index=False
)

print("Saved:")
print("- e1_nli_dev_oof_predictions.csv")
print("- e1_nli_dev_cv_results.csv")

## NLI Baseline - Results

Το πρώτο baseline χρησιμοποιεί μόνο τα 6 NLI features:
- src_ent, src_neu, src_con
- tgt_ent, tgt_neu, tgt_con

Τα καλύτερα αποτελέσματα εμφανίζονται στο MT, όπου το NLI μπορεί να εντοπίσει πιο καθαρές σημασιολογικές αποκλίσεις.  
Το DM έχει χαμηλότερο Accuracy, κάτι που δείχνει ότι πιθανόν χρειάζεται εξωτερική γνώση ή επιπλέον features.
Το PG έχει καλό Accuracy αλλά χαμηλότερο Spearman, άρα το μοντέλο ταξινομεί σωστά αρκετά δείγματα αλλά δεν αποτυπώνει τέλεια τη βαθμίδα αβεβαιότητας.

# 4 SEMANTIC SIMILARITY FEATURES


In [ ]:
#sentence embeddings library
%pip install sentence-transformers

In [ ]:
#Φορτώνουμε το μοντέλο που θα υπολογίσει τα sentence embeddings
from sentence_transformers import SentenceTransformer
import numpy as np

sim_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

In [ ]:
#cleaning text fields
text_cols = ["src", "hyp", "tgt"]

for col in text_cols:
    df[col] = df[col].fillna("").astype(str)

In [ ]:
#calculating embeddings για src, hyp, tgt
src_emb = sim_model.encode(
    df["src"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

hyp_emb = sim_model.encode(
    df["hyp"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

tgt_emb = sim_model.encode(
    df["tgt"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

In [ ]:
#Calculating the 3 basic similarity features: sim_src_hyp, sim_tgt_hyp, sim_src_tgtd
df["sim_src_hyp"] = np.sum(src_emb * hyp_emb, axis=1)
df["sim_tgt_hyp"] = np.sum(tgt_emb * hyp_emb, axis=1)
df["sim_src_tgt"] = np.sum(src_emb * tgt_emb, axis=1)

df[["sim_src_hyp", "sim_tgt_hyp", "sim_src_tgt"]].head()

In [ ]:
#Adding some similarity features
df["sim_max"] = df[["sim_src_hyp", "sim_tgt_hyp"]].max(axis=1)
df["sim_min"] = df[["sim_src_hyp", "sim_tgt_hyp"]].min(axis=1)
df["sim_mean"] = df[["sim_src_hyp", "sim_tgt_hyp"]].mean(axis=1)

df["sim_gap_tgt_src"] = df["sim_tgt_hyp"] - df["sim_src_hyp"]
df["sim_hyp_distance_from_truth"] = 1.0 - df["sim_tgt_hyp"]

In [ ]:
# Check and save development NLI + similarity features

similarity_feature_cols = [
    "sim_src_hyp",
    "sim_tgt_hyp",
    "sim_src_tgt",
    "sim_max",
    "sim_min",
    "sim_mean",
    "sim_gap_tgt_src",
    "sim_hyp_distance_from_truth"
]

print("Rows:", len(df))
print(
    "Missing similarity values:",
    df[similarity_feature_cols].isna().sum().sum()
)

print("\nSimilarity feature ranges:")
print(df[similarity_feature_cols].describe().loc[["min", "mean", "max"]])

assert len(df) == 499
assert df[similarity_feature_cols].isna().sum().sum() == 0

similarity_path = "nli_similarity_features_dev_agnostic.csv"
df.to_csv(similarity_path, index=False)

print("\nSaved:", similarity_path)
print("Shape:", df.shape)

## E2 - Semantic Similarity only baseline

Το πείραμα E2 χρησιμοποιεί αποκλειστικά semantic similarity features, χωρίς NLI features.  
Στόχος είναι να αξιολογηθεί αν η σημασιολογική ομοιότητα μεταξύ source, target και hypothesis επαρκεί ως ανεξάρτητο baseline για hallucination detection.

In [ ]:
# E2 - Semantic Similarity only feature set

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score
from scipy.stats import spearmanr
import numpy as np
import pandas as pd

e2_sim_cols = [
    "sim_src_hyp",
    "sim_tgt_hyp",
    "sim_src_tgt",
    "sim_max",
    "sim_min",
    "sim_mean",
    "sim_gap_tgt_src",
    "sim_hyp_distance_from_truth"
]

e2_X = df[e2_sim_cols].values
y_reg = df["p_hallucination"].values
y_bin = df["label_bin"].values
tasks = df["task"].values

print("E2 feature matrix shape:", e2_X.shape)
print("E2 features:")
for col in e2_sim_cols:
    print("-", col)

In [ ]:
# E2 - Random Forest with 5-fold cross-validation

e2_fold_results = []

e2_all_val_preds = []
e2_all_val_true_reg = []
e2_all_val_true_bin = []
e2_all_val_tasks = []

stratify_key = df["task"].astype(str) + "_" + df["label_bin"].astype(str)

for fold, (train_idx, val_idx) in enumerate(common_folds, start=1):

    X_train = e2_X[train_idx]
    X_val = e2_X[val_idx]

    y_train_reg = y_reg[train_idx]
    y_val_reg = y_reg[val_idx]
    y_val_bin = y_bin[val_idx]
    val_tasks = tasks[val_idx]

    model = RandomForestRegressor(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train, y_train_reg)

    val_preds = model.predict(X_val)
    val_preds = np.clip(val_preds, 0, 1)

    val_pred_bin = (val_preds >= THRESHOLD).astype(int)

    acc = accuracy_score(y_val_bin, val_pred_bin)
    sp = spearmanr(y_val_reg, val_preds).correlation

    e2_fold_results.append({
        "fold": fold,
        "accuracy": acc,
        "spearman": sp
    })

    e2_all_val_preds.extend(val_preds)
    e2_all_val_true_reg.extend(y_val_reg)
    e2_all_val_true_bin.extend(y_val_bin)
    e2_all_val_tasks.extend(val_tasks)

    print(f"Fold {fold} -> Accuracy: {acc:.4f}, Spearman: {sp:.4f}")

e2_results_df = pd.DataFrame(e2_fold_results)
e2_results_df

In [ ]:
# E2 - Overall out-of-fold results

e2_all_val_preds = np.array(e2_all_val_preds)
e2_all_val_true_reg = np.array(e2_all_val_true_reg)
e2_all_val_true_bin = np.array(e2_all_val_true_bin)
e2_all_val_tasks = np.array(e2_all_val_tasks)

e2_all_val_pred_bin = (e2_all_val_preds >= THRESHOLD).astype(int)

e2_overall_acc = accuracy_score(
    e2_all_val_true_bin,
    e2_all_val_pred_bin
)

e2_overall_sp = spearmanr(
    e2_all_val_true_reg,
    e2_all_val_preds
).correlation

print("E2 Mean CV Accuracy:", e2_results_df["accuracy"].mean())
print("E2 Mean CV Spearman:", e2_results_df["spearman"].mean())
print("E2 Overall OOF Accuracy:", e2_overall_acc)
print("E2 Overall OOF Spearman:", e2_overall_sp)

In [ ]:
# E2 - Per-task analysis

e2_oof_df = pd.DataFrame({
    "task": e2_all_val_tasks,
    "y_true_reg": e2_all_val_true_reg,
    "y_true_bin": e2_all_val_true_bin,
    "y_pred_reg": e2_all_val_preds
})

e2_oof_df["y_pred_bin"] = (e2_oof_df["y_pred_reg"] >= THRESHOLD).astype(int)

for task_name in sorted(e2_oof_df["task"].unique()):

    task_df = e2_oof_df[e2_oof_df["task"] == task_name]

    acc = accuracy_score(
        task_df["y_true_bin"],
        task_df["y_pred_bin"]
    )

    sp = spearmanr(
        task_df["y_true_reg"],
        task_df["y_pred_reg"]
    ).correlation

    print(f"{task_name} -> Accuracy: {acc:.4f}, Spearman: {sp:.4f}")

In [ ]:
e2_results_df.to_csv(
    "e2_similarity_dev_cv_results.csv",
    index=False
)

e2_oof_df.to_csv(
    "e2_similarity_dev_oof_predictions.csv",
    index=False
)

print("Saved:")
print("- e2_similarity_dev_cv_results.csv")
print("- e2_similarity_dev_oof_predictions.csv")

## E3 - NLI + Semantic Similarity

Το τρίτο πείραμα συνδυάζει τα έξι NLI features με τα οκτώ semantic similarity features. Στόχος είναι να εξεταστεί αν οι δύο ομάδες χαρακτηριστικών παρέχουν συμπληρωματική πληροφορία για την ανίχνευση hallucinations.

In [ ]:
#Define feature groups for the NLI + Semantic Similarity experiment

nli_cols = [
    "src_ent", "src_neu", "src_con",
    "tgt_ent", "tgt_neu", "tgt_con"
]

sim_cols = [
    "sim_src_hyp",
    "sim_tgt_hyp",
    "sim_src_tgt",
    "sim_max",
    "sim_min",
    "sim_mean",
    "sim_gap_tgt_src",
    "sim_hyp_distance_from_truth"
]

nli_sim_feature_cols  = nli_cols + sim_cols

nli_sim_X  = df[nli_sim_feature_cols ].values
y_reg = df["p_hallucination"].values
y_bin = df["label_bin"].values
tasks = df["task"].values

print("nli_sim_X shape:", nli_sim_X.shape)

In [ ]:
#Ελέγχουμε ότι το dataframe περιέχει ήδη τα NLI features.

print(df.shape)

df[[
    "src_ent", "src_neu", "src_con",
    "tgt_ent", "tgt_neu", "tgt_con"
]].head()

In [ ]:
# E3 - Random Forest με κοινό 5-fold cross-validation
# πάνω στα NLI + Semantic Similarity features

fold_results = []

all_val_preds = []
all_val_true_reg = []
all_val_true_bin = []
all_val_tasks = []

all_val_indices = []

for fold, (train_idx, val_idx) in enumerate(
    common_folds,
    start=1
):
    # Training και validation δεδομένα του fold
    X_train = nli_sim_X[train_idx]
    X_val = nli_sim_X[val_idx]

    y_train_reg = y_reg[train_idx]
    y_val_reg = y_reg[val_idx]

    y_val_bin = y_bin[val_idx]
    val_tasks = tasks[val_idx]

    model = RandomForestRegressor(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train, y_train_reg)

    val_pred = model.predict(X_val)
    val_pred = np.clip(val_pred, 0.0, 1.0)

    metrics = evaluate_predictions(
        y_true_bin=y_val_bin,
        y_true_prob=y_val_reg,
        y_pred_prob=val_pred,
        threshold=THRESHOLD
    )

    fold_results.append({
        "fold": fold,
        "accuracy": metrics["accuracy"],
        "spearman": metrics["spearman"]
    })

    all_val_preds.extend(val_pred)
    all_val_true_reg.extend(y_val_reg)
    all_val_true_bin.extend(y_val_bin)
    all_val_tasks.extend(val_tasks)
    all_val_indices.extend(val_idx)
    
nli_sim_results_df = pd.DataFrame(fold_results)

display(nli_sim_results_df)

In [ ]:
# E3 - Per-task analysis and indexed OOF predictions

nli_sim_oof_df = pd.DataFrame({
    "original_index": all_val_indices,
    "task": all_val_tasks,
    "y_true_reg": all_val_true_reg,
    "y_true_bin": all_val_true_bin,
    "y_pred_reg": all_val_preds
})

# Επαναφορά της αρχικής σειράς των development samples
nli_sim_oof_df = (
    nli_sim_oof_df
    .sort_values("original_index")
    .reset_index(drop=True)
)

nli_sim_oof_df["y_pred_bin"] = (
    nli_sim_oof_df["y_pred_reg"] >= THRESHOLD
).astype(int)

print("E3 results per task:")

for task_name in sorted(nli_sim_oof_df["task"].unique()):

    task_df = nli_sim_oof_df[
        nli_sim_oof_df["task"] == task_name
    ]

    task_accuracy = accuracy_score(
        task_df["y_true_bin"],
        task_df["y_pred_bin"]
    )

    task_spearman = spearmanr(
        task_df["y_true_reg"],
        task_df["y_pred_reg"]
    ).correlation

    print(
        f"{task_name} -> "
        f"Samples: {len(task_df)}, "
        f"Accuracy: {task_accuracy:.4f}, "
        f"Spearman: {task_spearman:.4f}"
    )

assert len(nli_sim_oof_df) == 499
assert nli_sim_oof_df["original_index"].nunique() == 499
assert nli_sim_oof_df["y_pred_reg"].notna().all()

print("\nE3 indexed OOF predictions are ready.")

In [ ]:
#Feature importance
final_model = RandomForestRegressor(
    n_estimators=500,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

final_model.fit(nli_sim_X, y_reg)

importance_df = pd.DataFrame({
    "feature": nli_sim_feature_cols,
    "importance": final_model.feature_importances_
}).sort_values("importance", ascending=False)

importance_df

In [ ]:
#Υπολογίζουμε τον μέσο όρο των αποτελεσμάτων από τα 5 folds.
print("Mean CV Accuracy:", nli_sim_results_df ["accuracy"].mean())
print("Mean CV Spearman:", nli_sim_results_df["spearman"].mean())

In [ ]:
#Υπολογίζουμε συνολικά αποτελέσματα πάνω σε όλες τις validation προβλέψεις.

all_val_preds = np.array(all_val_preds)
all_val_true_reg = np.array(all_val_true_reg)
all_val_true_bin = np.array(all_val_true_bin)
all_val_tasks = np.array(all_val_tasks)

all_val_pred_bin = (all_val_preds >= THRESHOLD).astype(int)

overall_acc = accuracy_score(all_val_true_bin, all_val_pred_bin)
overall_sp = spearmanr(all_val_true_reg, all_val_preds).correlation

print("Overall OOF Accuracy:", overall_acc)
print("Overall OOF Spearman:", overall_sp)

assert len(all_val_preds) == 499
assert len(all_val_true_reg) == 499
assert len(all_val_true_bin) == 499

In [ ]:
# Save E3 development-only results

from pathlib import Path

OUTPUT_DIR = Path(".")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

e3_cv_path = OUTPUT_DIR / "e3_nli_similarity_dev_cv_results.csv"
e3_oof_path = OUTPUT_DIR / "e3_nli_similarity_dev_oof_predictions.csv"

nli_sim_results_df.to_csv(e3_cv_path, index=False)
nli_sim_oof_df.to_csv(e3_oof_path, index=False)

print("Saved:")
print("-", e3_cv_path.resolve())
print("-", e3_oof_path.resolve())

print("\nCV results shape:", nli_sim_results_df.shape)
print("OOF predictions shape:", nli_sim_oof_df.shape)

assert len(nli_sim_results_df) == 5
assert len(nli_sim_oof_df) == 499
assert nli_sim_oof_df["y_pred_reg"].notna().all()

# 5. Data Augmentation

Σε αυτό το στάδιο δημιουργούμε συνθετικά training samples με weak supervision. Για κάθε unlabeled sample ζητάμε από το Gemini δύο εξόδους: μία faithful έξοδο που πρέπει να είναι σωστή παράφραση του reference και μία hallucinated έξοδο που πρέπει να είναι ρεαλιστική αλλά μη υποστηριζόμενη. Τα synthetic labels δεν θεωρούνται απόλυτα αξιόπιστα, οπότε στη συνέχεια εφαρμόζεται strict filtering και δημιουργείται balanced synthetic dataset για χρήση στο training.


In [ ]:
#install Gemini SDK
!pip install -q -U google-genai

In [ ]:
#imports and augmentation settings
#Κρατάμε τα augmentation imports και το βασικό config συγκεντρωμένα εδώ.
#Για μικρό/μεγαλύτερο run αλλάζουμε κυρίως SAMPLES_PER_TASK και BATCH_SIZE.

import os
import re
import json
import time
import datetime
from pathlib import Path
from getpass import getpass

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

from google import genai
from google.genai import types

#Αριθμός αρχικών samples που παίρνουμε από κάθε task (DM, MT, PG).
#Για δοκιμές κρατάμε μικρή τιμή. Για μεγαλύτερο run αυξάνεται σταδιακά.
SAMPLES_PER_TASK = 200

#Πόσα samples στέλνουμε σε κάθε Gemini request.
#Μικρό batch είναι πιο σταθερό, μεγάλο batch μειώνει τα API requests.
BATCH_SIZE = 2

#Μικρή αναμονή μετά από κάθε batch ώστε να μη χτυπάμε συνεχώς το API.
REQUEST_SLEEP = 3.0

#Fail-fast retries: δεν περιμένουμε πολλά λεπτά σε αποτυχημένο batch.
MAX_BATCH_RETRIES = 2
RETRY_SLEEP_SECONDS = 5

#Αν ένα batch αποτύχει πλήρως, το γράφουμε στο failed log και συνεχίζουμε.
SKIP_FAILED_BATCHES = True

#Gemini models.
#Το primary είναι το φθηνότερο/ελαφρύτερο μοντέλο για πολλά μικρά generations.
#Το fallback χρησιμοποιείται μόνο αν το primary έχει θέμα ποιότητας/διαθεσιμότητας.
AUG_MODEL_PRIMARY = "gemini-2.5-flash-lite"
AUG_MODEL_FALLBACK = "gemini-2.5-flash"

#Επιλογή μοντέλου για το τρέχον run.
USE_FALLBACK_MODEL = False
AUG_MODEL = AUG_MODEL_FALLBACK if USE_FALLBACK_MODEL else AUG_MODEL_PRIMARY

#Smoke test.
#Το κρατάμε False by default γιατί κάθε κλήση καταναλώνει quota.
RUN_SMOKE_TEST = False

# Output directory για checkpoints και synthetic data.
# Όλα τα intermediate/final αρχεία του augmentation γράφονται εδώ.
AUG_OUTPUT_DIR = Path(
    (__import__('os').environ.get('SHROOM_PROJECT_ROOT', '.') + '/data/6. Feature Extraction on Augmented Training Data')
)

#Output directory για checkpoints και synthetic data.
#Όλα τα intermediate/final αρχεία του augmentation γράφονται εδώ.
AUG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\n================================")
print("Augmentation Configuration:")
print(f"SAMPLES_PER_TASK: {SAMPLES_PER_TASK}")
print(f"BATCH_SIZE: {BATCH_SIZE}")
print(f"REQUEST_SLEEP: {REQUEST_SLEEP}s")
print(f"MAX_BATCH_RETRIES: {MAX_BATCH_RETRIES}")
print(f"RETRY_SLEEP_SECONDS: {RETRY_SLEEP_SECONDS}s")
print(f"SKIP_FAILED_BATCHES: {SKIP_FAILED_BATCHES}")
print(f"AUG_MODEL_PRIMARY: {AUG_MODEL_PRIMARY}")
print(f"AUG_MODEL_FALLBACK: {AUG_MODEL_FALLBACK}")
print(f"USE_FALLBACK_MODEL: {USE_FALLBACK_MODEL}")
print(f"Selected Gemini model: {AUG_MODEL}")
print("Augmentation output dir:", AUG_OUTPUT_DIR.resolve())
print("================================\n")


In [ ]:
#Gemini / Vertex AI setup with service account

from pathlib import Path
from google import genai
from google.genai import types
from google.oauth2 import service_account

#Path στο JSON key.
#Μην ανεβάσεις αυτό το αρχείο σε GitHub ή ChatGPT.
SERVICE_ACCOUNT_FILE = Path(__import__('os').environ['GOOGLE_APPLICATION_CREDENTIALS'])

PROJECT_ID = __import__('os').environ['GOOGLE_CLOUD_PROJECT']

#Συνήθως δουλεύει το us-central1 για Vertex AI Gemini.
#Αν δεν δουλέψει, ρώτα τον καθηγητή ποιο location έχει ενεργοποιηθεί.
LOCATION = "us-central1"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE,
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

gemini_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    credentials=credentials
)

print("Vertex AI Gemini client initialized.")
print("Project:", PROJECT_ID)
print("Location:", LOCATION)

In [ ]:
#Gemini smoke test
#Τρέξε το μόνο αν άλλαξες model/key και θες να ελέγξεις αν δουλεύει το API.
#Κάθε smoke test καταναλώνει quota.

if RUN_SMOKE_TEST:

    print("Running Gemini smoke test with model:", AUG_MODEL)

    test_response = gemini_client.models.generate_content(
        model=AUG_MODEL,
        contents='Return only this JSON: {"ok": true}',
        config=types.GenerateContentConfig(
            temperature=0.0,
            max_output_tokens=50,
            response_mime_type="application/json",
        ),
    )

    print(test_response.text)

else:
    print("Smoke test skipped. Set RUN_SMOKE_TEST = True only when needed.")

In [ ]:
#helper functions

#Επιστρέφει ποιο κείμενο πρέπει να χρησιμοποιηθεί ως reference.
#Για PG το reference είναι το src, γιατί το task είναι paraphrase generation.
#Για MT/DM το reference είναι το tgt, γιατί εκεί βρίσκεται η επιθυμητή έξοδος.
def get_reference_text(row):

    if "ref" in row.index and pd.notna(row["ref"]):
        ref_field = str(row["ref"]).strip()

        if ref_field == "src":
            return "" if pd.isna(row["src"]) else str(row["src"])

        if ref_field == "tgt":
            return "" if pd.isna(row["tgt"]) else str(row["tgt"])

    if row["task"] == "PG":
        return "" if pd.isna(row["src"]) else str(row["src"])

    return "" if pd.isna(row["tgt"]) else str(row["tgt"])


#Καθαρίζει απλό text field.
#Χρησιμοποιείται πριν μπουν texts στα prompts ή στα CSVs.
def clean_text_value(x):

    if pd.isna(x):
        return ""

    return str(x).strip()


#Προσπαθεί να κάνει parse την απάντηση του LLM ως JSON.
#Υποστηρίζει και JSON object και JSON array, γιατί το batch generation επιστρέφει λίστα.
def safe_parse_json(text):

    if text is None:
        return None

    try:
        return json.loads(text)
    except Exception:
        pass

    #Πρώτα δοκιμάζουμε JSON object.
    try:
        start = text.find("{")
        end = text.rfind("}") + 1

        if start >= 0 and end > start:
            json_text = text[start:end]
            return json.loads(json_text)

    except Exception:
        pass

    #Μετά δοκιμάζουμε JSON array, χρήσιμο για batch generation.
    try:
        start = text.find("[")
        end = text.rfind("]") + 1

        if start >= 0 and end > start:
            json_text = text[start:end]
            return json.loads(json_text)

    except Exception:
        return None

    return None


#Fallback token length για περίπτωση που δεν υπάρχει ήδη token_len από προηγούμενα cells.
if "token_len" not in globals():

    def token_len(text):
        return len(str(text).split())


In [ ]:
#function to build augmentation prompt
#Single-sample prompt για συμβατότητα/μελλοντική χρήση.
#Στο κύριο pipeline χρησιμοποιείται batch prompt, αλλά η λογική faithful/hallucinated είναι ίδια.

def build_augmentation_prompt(row):

    task = clean_text_value(row["task"])
    src = clean_text_value(row["src"])
    tgt = clean_text_value(row["tgt"])
    reference_text = get_reference_text(row)

    #Task-specific οδηγίες ώστε το Gemini να ξέρει ποιο reference πρέπει να ακολουθήσει.
    if task == "PG":
        task_instruction = (
            "Task: Paraphrase Generation. "
            "Use the source as the reference. "
        )

    elif task == "MT":
        task_instruction = (
            "Task: Machine Translation. "
            "Use the target as the reference translation. "
        )

    elif task == "DM":
        task_instruction = (
            "Task: Definition Modeling. "
            "Use the target as the correct definition. "
        )

    else:
        task_instruction = (
            "Task: Natural Language Generation. "
            "Use the reference as the semantic ground truth. "
        )

    prompt = f"""
Generate synthetic data for hallucination detection.

{task_instruction}

Source:
{src}

Target:
{tgt}

Reference:
{reference_text}

Return two short English outputs:
- faithful: a paraphrase of the reference, not an exact copy. It must preserve the same meaning but use different wording where possible. If a paraphrase is difficult, make a minimal wording change without adding or removing meaning.
- hallucinated: fluent and plausible, but unsupported, incorrect, or meaning-changing. The hallucination should be subtle, not absurd.

Do not explain.
Return only valid JSON with keys: faithful, hallucinated.
"""

    return prompt


In [ ]:
#Gemini structured output schemas and generation functions
#Τα schemas βοηθούν το Gemini να επιστρέφει σταθερό JSON που γίνεται εύκολα parse.

AUG_RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "faithful": {"type": "string"},
        "hallucinated": {"type": "string"}
    },
    "required": ["faithful", "hallucinated"]
}


BATCH_RESPONSE_SCHEMA = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "sample_id": {"type": "integer"},
            "faithful": {"type": "string"},
            "hallucinated": {"type": "string"}
        },
        "required": ["sample_id", "faithful", "hallucinated"]
    }
}


#helper function to extract text from Gemini response
#Χρησιμοποιείται γιατί μερικές φορές το response.text είναι κενό και το text βρίσκεται στα candidates/parts.
def extract_gemini_text(response):

    if getattr(response, "text", None) is not None:
        return response.text.strip()

    try:
        candidates = getattr(response, "candidates", None)

        if candidates and len(candidates) > 0:
            content = getattr(candidates[0], "content", None)
            parts = getattr(content, "parts", None)

            if parts:
                texts = []

                for part in parts:
                    part_text = getattr(part, "text", None)

                    if part_text is not None:
                        texts.append(part_text)

                if len(texts) > 0:
                    return "\n".join(texts).strip()

    except Exception:
        pass

    return None


#Κόβουμε πολύ μεγάλα texts ώστε το batch prompt να μένει μικρό και σταθερό.
def truncate_text(text, max_chars=1200):

    text = clean_text_value(text)

    if len(text) <= max_chars:
        return text

    return text[:max_chars] + "..."


#function to build batch augmentation prompt
#Εδώ φτιάχνουμε ένα prompt για πολλά samples μαζί, ώστε να μειώνονται τα API requests.
def build_batch_augmentation_prompt(batch_df):

    items = []

    for _, row in batch_df.iterrows():

        item = {
            "sample_id": int(row["sample_id"]),
            "task": clean_text_value(row["task"]),
            "src": truncate_text(row["src"]),
            "tgt": truncate_text(row["tgt"]),
            "reference": truncate_text(get_reference_text(row))
        }

        items.append(item)

    prompt = f"""
Generate synthetic training data for hallucination detection.

For each input item, create exactly two short English outputs:
1. faithful: a paraphrase of the reference, not an exact copy. It must preserve the same meaning but use different wording where possible. If a paraphrase is difficult, make a minimal wording change without adding or removing meaning.
2. hallucinated: fluent and plausible, but unsupported, incorrect, or meaning-changing. The hallucination should be subtle, not absurd.

Rules:
- Do not explain.
- Keep each output short.
- Return only valid JSON.
- Return one object per input item.
- Preserve the sample_id.

Input items:
{json.dumps(items, ensure_ascii=False, indent=2)}

Return JSON array with this schema:
[
  {{
    "sample_id": 0,
    "faithful": "...",
    "hallucinated": "..."
  }}
]
"""

    return prompt


#function to call Gemini for one synthetic pair with retries
#Το κρατάμε ως fallback/debugging, αλλά το κανονικό pipeline χρησιμοποιεί batch generation.
def generate_synthetic_pair(row, model=None, max_retries=None, retry_sleep=None):

    if model is None:
        model = AUG_MODEL

    if max_retries is None:
        max_retries = MAX_BATCH_RETRIES

    if retry_sleep is None:
        retry_sleep = RETRY_SLEEP_SECONDS

    prompt = build_augmentation_prompt(row)

    last_error = None

    for attempt in range(max_retries):

        try:
            response = gemini_client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.4,
                    max_output_tokens=1000,
                    response_mime_type="application/json",
                    response_schema=AUG_RESPONSE_SCHEMA,
                ),
            )

            raw_text = extract_gemini_text(response)

            if raw_text is None or raw_text.strip() == "":
                last_error = "empty_response"
                print(f"Empty Gemini response for {model} | attempt {attempt + 1}/{max_retries}")
                time.sleep(retry_sleep)
                continue

            parsed = safe_parse_json(raw_text)

            if parsed is None:
                last_error = raw_text[:500]
                print(f"JSON parse failed for {model} | attempt {attempt + 1}/{max_retries}")
                time.sleep(retry_sleep)
                continue

            return parsed, raw_text

        except Exception as e:

            last_error = str(e)
            error_text = str(e)

            if (
                "401" in error_text
                or "403" in error_text
                or "UNAUTHENTICATED" in error_text
                or "PERMISSION_DENIED" in error_text
                or "API key not valid" in error_text
            ):
                raise ValueError(
                    "Gemini credentials are invalid or do not have permission."
                ) from e

            print(f"Gemini single-sample error for {model} | attempt {attempt + 1}/{max_retries}")
            print(error_text[:300])
            time.sleep(retry_sleep)

    return None, last_error


#function to call Gemini for a batch of samples
#Fail-fast: λίγα retries και μικρή αναμονή. Αν αποτύχει, το batch γράφεται στο failed log.
def generate_synthetic_batch(batch_df, model=None, max_retries=None, retry_sleep=None):

    if model is None:
        model = AUG_MODEL

    if max_retries is None:
        max_retries = MAX_BATCH_RETRIES

    if retry_sleep is None:
        retry_sleep = RETRY_SLEEP_SECONDS

    prompt = build_batch_augmentation_prompt(batch_df)
    last_error = None

    for attempt in range(max_retries):

        try:
            response = gemini_client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.4,
                    max_output_tokens=5000,
                    response_mime_type="application/json",
                    response_schema=BATCH_RESPONSE_SCHEMA,
                ),
            )

            raw_text = extract_gemini_text(response)

            if raw_text is None or raw_text.strip() == "":
                last_error = "empty_response"
                print(f"Empty Gemini batch response for {model} | attempt {attempt + 1}/{max_retries}")
                time.sleep(retry_sleep)
                continue

            parsed = safe_parse_json(raw_text)

            if parsed is None:
                last_error = raw_text[:500]
                print(f"Batch JSON parse failed for {model} | attempt {attempt + 1}/{max_retries}")
                time.sleep(retry_sleep)
                continue

            if not isinstance(parsed, list):
                last_error = str(parsed)[:500]
                print(f"Batch response is not a list for {model} | attempt {attempt + 1}/{max_retries}")
                time.sleep(retry_sleep)
                continue

            return parsed, raw_text

        except Exception as e:

            last_error = str(e)
            error_text = str(e)

            if (
                "401" in error_text
                or "403" in error_text
                or "UNAUTHENTICATED" in error_text
                or "PERMISSION_DENIED" in error_text
                or "API key not valid" in error_text
            ):
                raise ValueError(
                    "Gemini credentials are invalid or do not have permission."
                ) from e

            if (
                "429" in error_text
                or "503" in error_text
                or "UNAVAILABLE" in error_text
                or "RESOURCE_EXHAUSTED" in error_text
                or "quota" in error_text.lower()
                or "high demand" in error_text.lower()
            ):
                print(f"Gemini temporary limit/high demand for {model} | attempt {attempt + 1}/{max_retries}")
                print(f"Waiting {retry_sleep} seconds...")
                time.sleep(retry_sleep)
                continue

            print(f"Gemini batch error for {model} | attempt {attempt + 1}/{max_retries}")
            print(error_text[:300])
            time.sleep(retry_sleep)

    return None, last_error


In [ ]:
#Choosing dataset for augmentation
#Εδώ επιλέγουμε μικρό balanced subset από το unlabeled train set.
#Το subset είναι balanced ανά task ώστε το augmentation να καλύπτει DM, MT και PG.

#Χρησιμοποιούμε μόνο unlabeled train data.
#Δεν χρησιμοποιούμε validation/test data για synthetic training data.
augmentation_base_df = train_agnostic_df.copy()

#Έλεγχος ότι υπάρχουν οι απαραίτητες στήλες.
required_cols = ["task", "src", "hyp", "tgt"]
missing_cols = [col for col in required_cols if col not in augmentation_base_df.columns]

if len(missing_cols) > 0:
    raise ValueError(f"Missing columns for augmentation: {missing_cols}")

#Κρατάμε stable sample id από το αρχικό index.
#Έτσι αν αλλάξεις SAMPLES_PER_TASK από 5 σε 20 ή 100 δεν μπερδεύεται το checkpoint.
augmentation_base_df = augmentation_base_df.reset_index().rename(columns={"index": "sample_id"})

#Καθαρίζουμε τα βασικά text fields.
for col in ["task", "src", "hyp", "tgt"]:
    augmentation_base_df[col] = augmentation_base_df[col].fillna("").astype(str).str.strip()

augmentation_base_df["sample_id"] = augmentation_base_df["sample_id"].astype(int)

#Κρατάμε μόνο τα τρία SHROOM tasks.
augmentation_base_df = augmentation_base_df[
    augmentation_base_df["task"].isin(["DM", "MT", "PG"])
].copy()

if augmentation_base_df.empty:
    raise ValueError("No rows left after task filtering.")

#Αν δεν υπάρχει ref, το δημιουργούμε με βάση το task.
if "ref" not in augmentation_base_df.columns:
    augmentation_base_df["ref"] = np.where(
        augmentation_base_df["task"] == "PG",
        "src",
        "tgt"
    )

#Κρατάμε balanced δείγμα ανά task με σταθερό random_state για αναπαραγωγιμότητα.
sample_parts = []

for task_name, task_df in augmentation_base_df.groupby("task"):

    n_samples = min(SAMPLES_PER_TASK, len(task_df))

    sampled_task_df = task_df.sample(
        n=n_samples,
        random_state=RANDOM_STATE
    )

    sample_parts.append(sampled_task_df)

sample_unlabeled = pd.concat(
    sample_parts,
    ignore_index=True
)

print("Sample for augmentation shape:", sample_unlabeled.shape)
print(sample_unlabeled["task"].value_counts().to_string())

display(sample_unlabeled.head())


In [ ]:
#Synthetic data generation with batch checkpoint/resume
#Κύριο generation loop: διαβάζει checkpoint, τρέχει μόνο τα remaining samples και σώζει πρόοδο μετά από κάθε batch.

raw_save_path = AUG_OUTPUT_DIR / "synthetic_raw_data.csv"
failed_save_path = AUG_OUTPUT_DIR / "synthetic_failed_generation.csv"
CHECKPOINT_MODE = "gemini_batch_v3"

#Recover προηγούμενο checkpoint, αν υπάρχει.
#Έτσι μπορούμε να σταματήσουμε/ξανατρέξουμε χωρίς να χαθεί η πρόοδος.
if raw_save_path.exists() and raw_save_path.stat().st_size > 0:

    try:
        synthetic_raw_df = pd.read_csv(raw_save_path)

        #Αν το checkpoint είναι από παλιό generation mode, το αγνοούμε για να μη μπερδευτούν sample_ids.
        if "generation_mode" in synthetic_raw_df.columns:
            synthetic_raw_df = synthetic_raw_df[
                synthetic_raw_df["generation_mode"] == CHECKPOINT_MODE
            ].copy()
        else:
            synthetic_raw_df = pd.DataFrame()

        if not synthetic_raw_df.empty and "sample_id" in synthetic_raw_df.columns:
            completed_sample_ids = set(
                synthetic_raw_df["sample_id"].dropna().astype(int).tolist()
            )
            synthetic_records = synthetic_raw_df.to_dict("records")
        else:
            completed_sample_ids = set()
            synthetic_records = []

    except pd.errors.EmptyDataError:
        synthetic_records = []
        completed_sample_ids = set()

else:
    synthetic_records = []
    completed_sample_ids = set()

failed_records = []

remaining_df = sample_unlabeled[
    ~sample_unlabeled["sample_id"].isin(completed_sample_ids)
].copy()

print("\n----------------------------------")
print("Starting/resuming batch synthetic data generation...")
print("Already completed:", len(completed_sample_ids))
print("Remaining samples:", len(remaining_df))
print("Batch size:", BATCH_SIZE)
print("Selected model:", AUG_MODEL)
print("----------------------------------")

start_time = time.time()

batch_starts = list(range(0, len(remaining_df), BATCH_SIZE))

for start_idx in tqdm(batch_starts, desc="Generating synthetic batches"):

    batch_df = remaining_df.iloc[start_idx:start_idx + BATCH_SIZE].copy()

    #Κρατάμε μόνο samples που έχουν source και reference.
    valid_rows = []

    for _, row in batch_df.iterrows():

        sample_id = int(row["sample_id"])
        src = clean_text_value(row["src"])
        reference_text = get_reference_text(row)

        if src == "" or reference_text == "":
            failed_records.append({
                "sample_id": sample_id,
                "task": clean_text_value(row["task"]),
                "reason": "empty_source_or_reference",
                "error_type": "data",
                "raw_response": "",
                "timestamp": datetime.datetime.now().isoformat()
            })
        else:
            valid_rows.append(row)

    if len(valid_rows) == 0:
        continue

    batch_df = pd.DataFrame(valid_rows)

    parsed_batch, response_or_error = generate_synthetic_batch(
        batch_df,
        model=AUG_MODEL,
        max_retries=MAX_BATCH_RETRIES,
        retry_sleep=RETRY_SLEEP_SECONDS
    )

    #Αν αποτύχει το batch, δοκιμάζουμε single-sample fallback.
    if parsed_batch is None:

        print("Batch failed. Trying single-sample fallback...")

        recovered_records = []

        for _, row in batch_df.iterrows():

            sample_id = int(row["sample_id"])

            parsed_single, raw_single = generate_synthetic_pair(
                row,
                model=AUG_MODEL,
                max_retries=1,
                retry_sleep=RETRY_SLEEP_SECONDS
            )

            if parsed_single is None:
                failed_records.append({
                    "sample_id": sample_id,
                    "task": clean_text_value(row["task"]),
                    "reason": "single_generation_failed_after_batch_failed",
                    "error_type": "generation",
                    "raw_response": response_or_error,
                    "timestamp": datetime.datetime.now().isoformat()
                })
                continue

            recovered_records.append((row, parsed_single))

        parsed_batch = []

        for row, parsed_single in recovered_records:
            parsed_batch.append({
                "sample_id": int(row["sample_id"]),
                "faithful": parsed_single.get("faithful", ""),
                "hallucinated": parsed_single.get("hallucinated", "")
            })

        if len(parsed_batch) == 0:
            pd.DataFrame(synthetic_records).to_csv(raw_save_path, index=False)

            if len(failed_records) > 0:
                pd.DataFrame(failed_records).to_csv(failed_save_path, index=False)

            time.sleep(REQUEST_SLEEP)
            continue

    parsed_by_id = {}

    for item in parsed_batch:

        try:
            parsed_sample_id = int(item.get("sample_id"))
            parsed_by_id[parsed_sample_id] = item
        except Exception:
            continue

    for _, row in batch_df.iterrows():

        sample_id = int(row["sample_id"])

        if sample_id not in parsed_by_id:
            failed_records.append({
                "sample_id": sample_id,
                "task": clean_text_value(row["task"]),
                "reason": "missing_sample_in_batch_response",
                "error_type": "parsing",
                "raw_response": "",
                "timestamp": datetime.datetime.now().isoformat()
            })
            continue

        item = parsed_by_id[sample_id]

        faithful = clean_text_value(item.get("faithful", ""))
        hallucinated = clean_text_value(item.get("hallucinated", ""))

        reference_text = get_reference_text(row)

        base_record = {
            "sample_id": sample_id,
            "task": clean_text_value(row["task"]),
            "src": clean_text_value(row["src"]),
            "tgt": clean_text_value(row["tgt"]),
            "reference_text": reference_text,
            "original_hyp": clean_text_value(row["hyp"]) if "hyp" in row.index else "",
            "synthetic": True,
            "generation_mode": CHECKPOINT_MODE
        }

        #Φάση Α: faithful output -> Not Hallucination
        if faithful != "":
            synthetic_records.append({
                **base_record,
                "hyp": faithful,
                "label": "Not Hallucination",
                "label_bin": 0,
                "p_hallucination": 0.0,
                "source_type": "synthetic_faithful"
            })

        #Φάση Β: hallucinated output -> Hallucination
        if hallucinated != "":
            synthetic_records.append({
                **base_record,
                "hyp": hallucinated,
                "label": "Hallucination",
                "label_bin": 1,
                "p_hallucination": 1.0,
                "source_type": "synthetic_hallucination"
            })

    #Checkpoint μετά από κάθε batch.
    pd.DataFrame(synthetic_records).to_csv(raw_save_path, index=False)

    if len(failed_records) > 0:
        pd.DataFrame(failed_records).to_csv(failed_save_path, index=False)

    time.sleep(REQUEST_SLEEP)

end_time = time.time()

synthetic_raw_df = pd.DataFrame(synthetic_records)
synthetic_df = synthetic_raw_df.copy()

print("\n----------------------------------")
print(f"Χρόνος παραγωγής: {(end_time - start_time) / 60:.2f} λεπτά")
print("Generated synthetic shape:", synthetic_raw_df.shape)
print("Failed generations:", len(failed_records))
print("----------------------------------")

if not synthetic_raw_df.empty:
    display(synthetic_raw_df.head())
else:
    print("No synthetic samples were generated.")


In [ ]:
#Load semantic similarity model if needed

if "sim_model" not in globals():
    from sentence_transformers import SentenceTransformer

    SIM_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
    sim_model = SentenceTransformer(SIM_MODEL_NAME)

    print("Semantic similarity model loaded:", SIM_MODEL_NAME)
else:
    print("Using existing semantic similarity model.")


In [ ]:
### Strict Filtering Phase

print("\n----------------------------------")
print("Filtering synthetic data...")
print("----------------------------------")

#Ξεκινάμε από το raw synthetic data που παρήχθη στο προηγούμενο βήμα.
synthetic_df = synthetic_raw_df.copy()
print("Before filtering:", synthetic_df.shape)

#Σταθερή δομή για τα τελικά synthetic data.
final_columns = [
    "sample_id", "task", "src", "tgt", "hyp", "label", "label_bin",
    "p_hallucination", "source_type", "synthetic", "reference_text",
    "original_hyp", "generation_mode", "sim_ref_hyp"
]

#Αν δεν υπάρχουν synthetic records, δημιουργούμε κενά DataFrames ώστε να μη σπάσουν τα επόμενα cells.
if synthetic_df.empty:
    print("\nNo synthetic data generated. Creating empty final_synthetic_df.")
    final_synthetic_df = pd.DataFrame(columns=final_columns)
    final_synthetic_df_balanced = final_synthetic_df.copy()

else:
    #Βεβαιωνόμαστε ότι υπάρχει reference_text.
    #Για PG reference = src, ενώ για DM/MT reference = tgt.
    if "reference_text" not in synthetic_df.columns:
        synthetic_df["reference_text"] = synthetic_df.apply(get_reference_text, axis=1)

    #Καθαρίζουμε βασικά πεδία πριν από τα filters.
    for col in ["src", "tgt", "hyp", "reference_text"]:
        synthetic_df[col] = synthetic_df[col].fillna("").astype(str).str.strip()

    #Αφαιρούμε άδεια hypotheses.
    synthetic_df = synthetic_df[
        synthetic_df["hyp"] != ""
    ].copy()

    #Υπολογισμός απλού token length για basic sanity checks.
    synthetic_df["hyp_tokens"] = synthetic_df["hyp"].apply(token_len)
    synthetic_df["reference_tokens"] = synthetic_df["reference_text"].apply(token_len)

    #Basic filtering βάσει μήκους.
    #Πετάμε outputs που είναι υπερβολικά μικρά ή υπερβολικά μεγάλα.
    synthetic_df = synthetic_df[
        (synthetic_df["hyp_tokens"] >= 3)
        & (synthetic_df["hyp_tokens"] <= 200)
    ].copy()

    #Exact-copy diagnostics πριν αφαιρέσουμε τα αντίγραφα.
    #Αυτό δείχνει αν το faithful generation αντιγράφει απλώς το reference.
    exact_copy_mask = (
        synthetic_df["hyp"].str.lower().str.strip()
        == synthetic_df["reference_text"].str.lower().str.strip()
    )

    print("\nExact-copy diagnostics:")
    print("Total exact copies:", int(exact_copy_mask.sum()))

    if "source_type" in synthetic_df.columns:
        print("Exact copies by source_type:")
        print(synthetic_df[exact_copy_mask]["source_type"].value_counts(dropna=False).to_string())

    #Αφαιρούμε outputs που είναι ακριβώς ίδια με το reference.
    #Δεν προσφέρουν πραγματικό augmentation.
    synthetic_df = synthetic_df[~exact_copy_mask].copy()

    #Αφαιρούμε outputs που είναι υπερβολικά μεγαλύτερα από το reference.
    synthetic_df = synthetic_df[
        synthetic_df["hyp_tokens"] <= 3 * synthetic_df["reference_tokens"].clip(lower=1)
    ].copy()

    print("After basic filtering:", synthetic_df.shape)

    if synthetic_df.empty:
        print("\nNo data after basic filtering. Creating empty final_synthetic_df.")
        final_synthetic_df = pd.DataFrame(columns=final_columns)
        final_synthetic_df_balanced = final_synthetic_df.copy()

    else:
        #Semantic similarity filtering.
        #Χρησιμοποιούμε reference_text και όχι πάντα src, ώστε το filtering να είναι σωστό για όλα τα tasks.
        syn_ref_emb = sim_model.encode(
            synthetic_df["reference_text"].tolist(),
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True
        )

        syn_hyp_emb = sim_model.encode(
            synthetic_df["hyp"].tolist(),
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True
        )

        #Cosine similarity, επειδή τα embeddings είναι normalized.
        synthetic_df["sim_ref_hyp"] = np.sum(
            syn_ref_emb * syn_hyp_emb,
            axis=1
        )

        #Thresholds σύμφωνα με τη λογική του οδηγού.
        #Hallucination: αρκετά κοντά ώστε να είναι ρεαλιστικό, όχι τόσο κοντά ώστε να είναι faithful.
        #Not Hallucination: υψηλή similarity με το reference.
        HALLUCINATION_MIN_SIM = 0.30
        HALLUCINATION_MAX_SIM = 0.95

        FAITHFUL_MIN_SIM = 0.75
        FAITHFUL_MAX_SIM = 0.99

        hallucination_mask = (
            (synthetic_df["label_bin"] == 1)
            & (synthetic_df["sim_ref_hyp"] >= HALLUCINATION_MIN_SIM)
            & (synthetic_df["sim_ref_hyp"] <= HALLUCINATION_MAX_SIM)
        )

        faithful_mask = (
            (synthetic_df["label_bin"] == 0)
            & (synthetic_df["sim_ref_hyp"] >= FAITHFUL_MIN_SIM)
            & (synthetic_df["sim_ref_hyp"] <= FAITHFUL_MAX_SIM)
        )

        final_synthetic_df = synthetic_df[
            hallucination_mask | faithful_mask
        ].copy()

        print("After semantic filtering:", final_synthetic_df.shape)

        print("\nLabel distribution:")
        print(final_synthetic_df["label"].value_counts(dropna=False).to_string())

        print("\nSimilarity statistics by source_type:")
        if not final_synthetic_df.empty and "source_type" in final_synthetic_df.columns:
            print(
                final_synthetic_df
                .groupby("source_type")["sim_ref_hyp"]
                .describe()
                .round(3)
                .to_string()
            )

        #Δημιουργούμε balanced version για training.
        #Το filtered dataset μπορεί να είναι ανισόρροπο, επειδή το filtering πετάει περισσότερα faithful samples.
        label_counts = final_synthetic_df["label"].value_counts(dropna=False)

        if len(label_counts) == 2:
            min_count = label_counts.min()

            final_synthetic_df_balanced = (
                final_synthetic_df
                .groupby("label", group_keys=False)
                .apply(lambda x: x.sample(n=min_count, random_state=RANDOM_STATE))
                .reset_index(drop=True)
            )
        else:
            print("\nWarning: only one label exists after filtering. Balanced dataset will be identical to filtered dataset.")
            final_synthetic_df_balanced = final_synthetic_df.copy()

print("\n----------------------------------")
print("Filtered synthetic shape:", final_synthetic_df.shape)
print("Balanced synthetic shape:", final_synthetic_df_balanced.shape)

if not final_synthetic_df.empty:
    display(final_synthetic_df.head())
else:
    print("Final synthetic DataFrame is empty. Skipping display.")
print("----------------------------------")


In [ ]:
#Manual inspection sample

#Παίρνουμε λίγα δείγματα για χειροκίνητο έλεγχο ποιότητας.
#Αν το filtered synthetic dataset είναι άδειο, δεν κάνουμε inspection.

if "final_synthetic_df" not in globals():
    print("final_synthetic_df does not exist. Run the filtering cell first.")

elif final_synthetic_df.empty:
    print("No synthetic data to inspect (final_synthetic_df is empty).")
    print("Check synthetic_raw_data.csv, failed generations, and filtering thresholds before continuing.")

else:
    inspection_cols = [
        "task",
        "label",
        "source_type",
        "sim_ref_hyp",
        "reference_text",
        "hyp"
    ]

    available_cols = [
        col for col in inspection_cols
        if col in final_synthetic_df.columns
    ]

    inspection_parts = []

    for _, group_df in final_synthetic_df.groupby(["task", "label"]):

        n_samples = min(3, len(group_df))

        if n_samples > 0:
            inspection_parts.append(
                group_df.sample(n=n_samples, random_state=RANDOM_STATE)
            )

    if len(inspection_parts) == 0:
        print("No synthetic data to inspect after grouping.")

    else:
        inspection_df = pd.concat(
            inspection_parts,
            ignore_index=True
        )

        if len(available_cols) == 0:
            print("No inspection columns are available.")
            print("Available columns:")
            print(final_synthetic_df.columns.tolist())

        else:
            display(inspection_df[available_cols])


In [ ]:
#Saving the final synthetic dataset

#Σώζουμε και το filtered dataset και το balanced dataset.
#Για training χρησιμοποιούμε συνήθως το balanced version.

if "final_synthetic_df" not in globals():
    raise ValueError("final_synthetic_df does not exist. Run the filtering cell first.")

expected_synthetic_cols = [
    "sample_id",
    "task",
    "src",
    "tgt",
    "reference_text",
    "original_hyp",
    "hyp",
    "label",
    "label_bin",
    "p_hallucination",
    "source_type",
    "synthetic",
    "generation_mode",
    "sim_ref_hyp"
]

#Αν το filtered dataframe είναι άδειο, κρατάμε τις αναμενόμενες στήλες.
if final_synthetic_df.empty:
    print("final_synthetic_df is empty. Saving empty filtered/balanced files.")
    final_synthetic_df = pd.DataFrame(columns=expected_synthetic_cols)

#Αν λείπουν στήλες από το filtered dataframe, τις προσθέτουμε.
for col in expected_synthetic_cols:
    if col not in final_synthetic_df.columns:
        final_synthetic_df[col] = np.nan

#Δημιουργούμε ξανά balanced dataframe με ασφαλή τρόπο.
if not final_synthetic_df.empty and "label" in final_synthetic_df.columns:

    label_counts = final_synthetic_df["label"].value_counts(dropna=False)

    if len(label_counts) == 2:
        min_count = label_counts.min()

        final_synthetic_df_balanced = (
            final_synthetic_df
            .groupby("label", group_keys=False)
            .apply(lambda x: x.sample(n=min_count, random_state=RANDOM_STATE))
            .reset_index(drop=True)
        )

    else:
        print("Warning: only one label exists after filtering.")
        print("Balanced dataset will be identical to filtered dataset.")
        final_synthetic_df_balanced = final_synthetic_df.copy()

else:
    print("Warning: no valid filtered synthetic samples.")
    final_synthetic_df_balanced = pd.DataFrame(columns=expected_synthetic_cols)

#Βεβαιωνόμαστε ότι και το balanced dataframe έχει τις αναμενόμενες στήλες.
for col in expected_synthetic_cols:
    if col not in final_synthetic_df_balanced.columns:
        final_synthetic_df_balanced[col] = np.nan

final_synthetic_path = AUG_OUTPUT_DIR / "filtered_synthetic_data.csv"
final_synthetic_jsonl_path = AUG_OUTPUT_DIR / "filtered_synthetic_data.jsonl"

final_synthetic_balanced_path = AUG_OUTPUT_DIR / "filtered_synthetic_data_balanced.csv"
final_synthetic_balanced_jsonl_path = AUG_OUTPUT_DIR / "filtered_synthetic_data_balanced.jsonl"

#Βεβαιωνόμαστε ότι το label υπάρχει σωστά και στο balanced dataset.
if "label" not in final_synthetic_df_balanced.columns or final_synthetic_df_balanced["label"].isna().any():
    final_synthetic_df_balanced["label"] = final_synthetic_df_balanced["label_bin"].map({
        1: "Hallucination",
        0: "Not Hallucination"
    })

#Βεβαιωνόμαστε ότι το label υπάρχει σωστά και στο filtered dataset.
if "label" not in final_synthetic_df.columns or final_synthetic_df["label"].isna().any():
    final_synthetic_df["label"] = final_synthetic_df["label_bin"].map({
        1: "Hallucination",
        0: "Not Hallucination"
    })
    
final_synthetic_df.to_csv(final_synthetic_path, index=False)
final_synthetic_df.to_json(
    final_synthetic_jsonl_path,
    orient="records",
    lines=True,
    force_ascii=False
)

final_synthetic_df_balanced.to_csv(final_synthetic_balanced_path, index=False)
final_synthetic_df_balanced.to_json(
    final_synthetic_balanced_jsonl_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("\n" + "="*60)
print("FINAL SYNTHETIC DATA SUMMARY")
print("="*60)

print("\nFiltered Dataset (final_synthetic_df):")
print(f"  Shape: {final_synthetic_df.shape}")

if not final_synthetic_df.empty and "label" in final_synthetic_df.columns:
    print("  Label distribution:")
    print(final_synthetic_df["label"].value_counts(dropna=False).to_string())
else:
    print("  No labels available or dataset is empty.")

print("\nBalanced Dataset (final_synthetic_df_balanced):")
print(f"  Shape: {final_synthetic_df_balanced.shape}")

if not final_synthetic_df_balanced.empty and "label" in final_synthetic_df_balanced.columns:
    print("  Label distribution:")
    print(final_synthetic_df_balanced["label"].value_counts(dropna=False).to_string())
else:
    print("  No labels available or dataset is empty.")

print("\n" + "-"*60)
print("Saved Files:")
print("-"*60)
print(f"Filtered CSV:        {final_synthetic_path.resolve()}")
print(f"Filtered JSONL:      {final_synthetic_jsonl_path.resolve()}")
print(f"Balanced CSV:        {final_synthetic_balanced_path.resolve()}")
print(f"Balanced JSONL:      {final_synthetic_balanced_jsonl_path.resolve()}")
print("="*60)

if not final_synthetic_df.empty:
    display(final_synthetic_df.head())
else:
    print("(Final synthetic DataFrame is empty)")

In [ ]:
#check: Θα μας πει αν το filtering κράτησε καλά δείγματα.
filtered_df = pd.read_csv(AUG_OUTPUT_DIR / "filtered_synthetic_data.csv")
balanced_df = pd.read_csv(AUG_OUTPUT_DIR / "filtered_synthetic_data_balanced.csv")

print("Filtered shape:", filtered_df.shape)
print(filtered_df["label"].value_counts(dropna=False) if "label" in filtered_df.columns else "No label column")

print("\nBalanced shape:", balanced_df.shape)
print(balanced_df["label"].value_counts(dropna=False) if "label" in balanced_df.columns else "No label column")

## E4 - NLI + Semantic Similarity + Synthetic Data

Στο τέταρτο πείραμα διατηρούνται τα 14 χαρακτηριστικά του E3 και
προστίθενται τα φιλτραρισμένα synthetic δείγματα μόνο στο training
μέρος κάθε fold. Τα validation δεδομένα περιλαμβάνουν αποκλειστικά
ανθρώπινα δείγματα του development set.

In [ ]:
# E4 - Load synthetic data and remove exact overlaps

# Το AUG_OUTPUT_DIR έχει ήδη οριστεί στο augmentation settings cell.
balanced_path = (
    AUG_OUTPUT_DIR
    / "filtered_synthetic_data_balanced.csv"
)

print("Path:", balanced_path)
print("Exists:", balanced_path.exists())

if not balanced_path.exists():
    raise FileNotFoundError(
        f"Το αρχείο δεν βρέθηκε: {balanced_path.resolve()}"
    )

# Φόρτωση των 518 balanced synthetic δειγμάτων
synthetic_e4_df = pd.read_csv(balanced_path)

print(
    "\nBalanced synthetic rows before overlap removal:",
    len(synthetic_e4_df)
)

required_cols = ["task", "src", "tgt", "hyp"]

missing_cols = [
    col
    for col in required_cols
    if col not in synthetic_e4_df.columns
]

if missing_cols:
    raise ValueError(
        f"Λείπουν οι απαραίτητες στήλες: {missing_cols}"
    )


# Δημιουργεί ένα κοινό κλειδί για τη σύγκριση των κειμένων
def make_text_key(input_df):

    key_cols = ["task", "src", "tgt", "hyp"]
    normalized_df = input_df[key_cols].copy()

    for col in key_cols:
        normalized_df[col] = (
            normalized_df[col]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
        )

    return normalized_df.agg(" ||| ".join, axis=1)


# Κλειδιά synthetic, development και official test
synthetic_keys = make_text_key(synthetic_e4_df)
development_keys = set(make_text_key(df))
test_keys = set(make_text_key(test_agnostic_df))

# Έλεγχος ακριβών επικαλύψεων
development_overlap_mask = synthetic_keys.isin(
    development_keys
)

test_overlap_mask = synthetic_keys.isin(
    test_keys
)

remove_overlap_mask = (
    development_overlap_mask
    | test_overlap_mask
)

print(
    "Exact overlaps with development:",
    int(development_overlap_mask.sum())
)

print(
    "Exact overlaps with official test:",
    int(test_overlap_mask.sum())
)

print(
    "Unique rows removed:",
    int(remove_overlap_mask.sum())
)

# Αφαίρεση των επικαλύψεων
synthetic_e4_df = (
    synthetic_e4_df.loc[~remove_overlap_mask]
    .copy()
    .reset_index(drop=True)
)

print(
    "\nSynthetic rows after overlap removal:",
    len(synthetic_e4_df)
)

# Αποθήκευση καθαρισμένου synthetic συνόλου
clean_synthetic_path = (
    AUG_OUTPUT_DIR
    / "e4_synthetic_clean.csv"
)

synthetic_e4_df.to_csv(
    clean_synthetic_path,
    index=False
)

print("Saved:", clean_synthetic_path)

In [ ]:
# E4 - Load existing augmented feature dataset
# and keep only the 518 synthetic samples

augmented_features_path = (
    AUG_OUTPUT_DIR
    / "nli_similarity_features_dataset.csv"
)

print("Feature file:", augmented_features_path)
print("Exists:", augmented_features_path.exists())

if not augmented_features_path.exists():
    raise FileNotFoundError(
        f"Το αρχείο δεν βρέθηκε: {augmented_features_path}"
    )

augmented_features_df = pd.read_csv(
    augmented_features_path
)

print("\nFull augmented feature shape:",
      augmented_features_df.shape)

print("\nSource type distribution:")
print(
    augmented_features_df["source_type"]
    .value_counts(dropna=False)
)


# Κρατάμε μόνο τις synthetic γραμμές.
synthetic_features_df = (
    augmented_features_df[
        augmented_features_df["source_type"].isin([
            "synthetic_hallucination",
            "synthetic_faithful"
        ])
    ]
    .copy()
    .reset_index(drop=True)
)


e4_feature_cols = [
    "src_ent",
    "src_neu",
    "src_con",
    "tgt_ent",
    "tgt_neu",
    "tgt_con",
    "sim_src_hyp",
    "sim_tgt_hyp",
    "sim_src_tgt",
    "sim_max",
    "sim_min",
    "sim_mean",
    "sim_gap_tgt_src",
    "sim_hyp_distance_from_truth"
]


print("\nSynthetic feature rows:",
      len(synthetic_features_df))

print("Synthetic feature matrix shape:",
      synthetic_features_df[e4_feature_cols].shape)

print(
    "Missing feature values:",
    synthetic_features_df[e4_feature_cols]
    .isna()
    .sum()
    .sum()
)

print("\nSynthetic target distribution:")
print(
    synthetic_features_df["p_hallucination"]
    .value_counts()
    .sort_index()
)


assert len(synthetic_features_df) == 518
assert synthetic_features_df[e4_feature_cols].shape == (518, 14)
assert (
    synthetic_features_df[e4_feature_cols]
    .isna()
    .sum()
    .sum()
    == 0
)

In [ ]:
# E4 - Prepare human and synthetic feature matrices

# Human development data: τα ίδια 14 features με το E3
e4_human_X = df[e4_feature_cols].to_numpy(dtype=float)

e4_human_y_reg = (
    df["p_hallucination"]
    .to_numpy(dtype=float)
)

e4_human_y_bin = (
    df["label_bin"]
    .to_numpy(dtype=int)
)

e4_human_tasks = (
    df["task"]
    .astype(str)
    .to_numpy()
)


# Synthetic data: χρησιμοποιούνται μόνο στο training
e4_synthetic_X = (
    synthetic_features_df[e4_feature_cols]
    .to_numpy(dtype=float)
)

e4_synthetic_y_reg = (
    synthetic_features_df["p_hallucination"]
    .to_numpy(dtype=float)
)


print("Human development matrix:", e4_human_X.shape)
print("Human regression target:", e4_human_y_reg.shape)
print("Human binary target:", e4_human_y_bin.shape)

print("\nSynthetic training matrix:", e4_synthetic_X.shape)
print("Synthetic target:", e4_synthetic_y_reg.shape)


# Έλεγχοι
assert e4_human_X.shape == (499, 14)
assert e4_synthetic_X.shape == (518, 14)

assert len(e4_human_y_reg) == 499
assert len(e4_human_y_bin) == 499
assert len(e4_human_tasks) == 499

assert len(e4_synthetic_y_reg) == 518

assert not np.isnan(e4_human_X).any()
assert not np.isnan(e4_synthetic_X).any()

# Επιβεβαίωση ότι τα ανθρώπινα features είναι ίδια με του E3
assert np.allclose(e4_human_X, nli_sim_X)

print("\nE4 matrices are ready.")

In [ ]:
# E4 - Random Forest with synthetic data added only to training folds

e4_fold_results = []

e4_all_val_indices = []
e4_all_val_preds = []
e4_all_val_true_reg = []
e4_all_val_true_bin = []
e4_all_val_tasks = []

assert len(common_folds) == 5


for fold, (train_idx, val_idx) in enumerate(
    common_folds,
    start=1
):

    # Human development samples του training μέρους
    human_X_train = e4_human_X[train_idx]
    human_y_train = e4_human_y_reg[train_idx]

    # Προσθήκη των 518 synthetic samples μόνο στο training
    X_train_augmented = np.vstack([
        human_X_train,
        e4_synthetic_X
    ])

    y_train_augmented = np.concatenate([
        human_y_train,
        e4_synthetic_y_reg
    ])

    # Το validation περιλαμβάνει μόνο human development samples
    X_val = e4_human_X[val_idx]
    y_val_reg = e4_human_y_reg[val_idx]
    y_val_bin = e4_human_y_bin[val_idx]
    val_tasks = e4_human_tasks[val_idx]

    model = RandomForestRegressor(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(
        X_train_augmented,
        y_train_augmented
    )

    val_pred = model.predict(X_val)
    val_pred = np.clip(val_pred, 0.0, 1.0)

    metrics = evaluate_predictions(
        y_true_bin=y_val_bin,
        y_true_prob=y_val_reg,
        y_pred_prob=val_pred,
        threshold=THRESHOLD
    )

    e4_fold_results.append({
        "fold": fold,
        "human_training_samples": len(train_idx),
        "synthetic_training_samples": len(e4_synthetic_X),
        "total_training_samples": len(X_train_augmented),
        "validation_samples": len(val_idx),
        "accuracy": metrics["accuracy"],
        "spearman": metrics["spearman"]
    })

    e4_all_val_indices.extend(val_idx)
    e4_all_val_preds.extend(val_pred)
    e4_all_val_true_reg.extend(y_val_reg)
    e4_all_val_true_bin.extend(y_val_bin)
    e4_all_val_tasks.extend(val_tasks)

    print(
        f"Fold {fold} -> "
        f"human train={len(train_idx)}, "
        f"synthetic train={len(e4_synthetic_X)}, "
        f"total train={len(X_train_augmented)}, "
        f"validation={len(val_idx)}, "
        f"Accuracy={metrics['accuracy']:.4f}, "
        f"Spearman={metrics['spearman']:.4f}"
    )


e4_results_df = pd.DataFrame(e4_fold_results)

display(e4_results_df)


# Βασικοί έλεγχοι
assert len(e4_results_df) == 5
assert len(e4_all_val_preds) == 499
assert len(e4_all_val_indices) == 499
assert len(set(e4_all_val_indices)) == 499

print("\nE4 cross-validation completed successfully.")

In [ ]:
# E4 - Overall OOF and per-task evaluation

e4_oof_df = pd.DataFrame({
    "original_index": e4_all_val_indices,
    "task": e4_all_val_tasks,
    "y_true_reg": e4_all_val_true_reg,
    "y_true_bin": e4_all_val_true_bin,
    "y_pred_reg": e4_all_val_preds
})

# Επαναφέρουμε τη σειρά του αρχικού development dataset
e4_oof_df = (
    e4_oof_df
    .sort_values("original_index")
    .reset_index(drop=True)
)

# Μετατροπή της συνεχούς πρόβλεψης σε δυαδική
e4_oof_df["y_pred_bin"] = (
    e4_oof_df["y_pred_reg"] >= THRESHOLD
).astype(int)


# Μέσος όρος των πέντε folds
e4_mean_cv_accuracy = e4_results_df["accuracy"].mean()
e4_mean_cv_spearman = e4_results_df["spearman"].mean()


# Συνολικά OOF αποτελέσματα
e4_overall_accuracy = accuracy_score(
    e4_oof_df["y_true_bin"],
    e4_oof_df["y_pred_bin"]
)

e4_overall_spearman = spearmanr(
    e4_oof_df["y_true_reg"],
    e4_oof_df["y_pred_reg"]
).correlation


print("E4 Mean CV Accuracy:", e4_mean_cv_accuracy)
print("E4 Mean CV Spearman:", e4_mean_cv_spearman)

print("\nE4 Overall OOF Accuracy:", e4_overall_accuracy)
print("E4 Overall OOF Spearman:", e4_overall_spearman)


# Αποτελέσματα ανά task
e4_task_results = []

print("\nE4 results per task:")

for task_name in sorted(e4_oof_df["task"].unique()):

    task_df = e4_oof_df[
        e4_oof_df["task"] == task_name
    ]

    task_accuracy = accuracy_score(
        task_df["y_true_bin"],
        task_df["y_pred_bin"]
    )

    task_spearman = spearmanr(
        task_df["y_true_reg"],
        task_df["y_pred_reg"]
    ).correlation

    e4_task_results.append({
        "task": task_name,
        "samples": len(task_df),
        "accuracy": task_accuracy,
        "spearman": task_spearman
    })

    print(
        f"{task_name} -> "
        f"Samples: {len(task_df)}, "
        f"Accuracy: {task_accuracy:.4f}, "
        f"Spearman: {task_spearman:.4f}"
    )


e4_task_results_df = pd.DataFrame(e4_task_results)

display(e4_task_results_df)


# Τελικοί έλεγχοι
assert len(e4_oof_df) == 499
assert e4_oof_df["original_index"].nunique() == 499
assert e4_oof_df["y_pred_reg"].notna().all()
assert e4_oof_df["y_pred_bin"].isin([0, 1]).all()

print("\nE4 OOF evaluation completed successfully.")

In [ ]:
# E4 - Save development-only results

e4_cv_path = "e4_synthetic_dev_cv_results.csv"
e4_oof_path = "e4_synthetic_dev_oof_predictions.csv"
e4_task_path = "e4_synthetic_dev_task_results.csv"

e4_results_df.to_csv(
    e4_cv_path,
    index=False
)

e4_oof_df.to_csv(
    e4_oof_path,
    index=False
)

e4_task_results_df.to_csv(
    e4_task_path,
    index=False
)

print("Saved:")
print("-", e4_cv_path)
print("-", e4_oof_path)
print("-", e4_task_path)

print("\nCV results shape:", e4_results_df.shape)
print("OOF predictions shape:", e4_oof_df.shape)
print("Task results shape:", e4_task_results_df.shape)

assert len(e4_results_df) == 5
assert len(e4_oof_df) == 499
assert len(e4_task_results_df) == 3

In [ ]:
# Final development-only experiment summary

experiment_summary_df = pd.DataFrame([
    {
        "experiment": "E1_NLI_only",
        "selection_eligible": "Yes",
        "overall_oof_accuracy": 0.779559,
        "overall_oof_spearman": 0.690455
    },
    {
        "experiment": "E2_similarity_only",
        "selection_eligible": "Yes",
        "overall_oof_accuracy": 0.705411,
        "overall_oof_spearman": 0.561303
    },
    {
        "experiment": "E3_NLI_similarity",
        "selection_eligible": "Yes",
        "overall_oof_accuracy": 0.807615,
        "overall_oof_spearman": 0.738835
    },
    {
        "experiment": "E4_synthetic_diagnostic",
        "selection_eligible": "No",
        "overall_oof_accuracy": e4_overall_accuracy,
        "overall_oof_spearman": e4_overall_spearman
    }
])

display(experiment_summary_df)

experiment_summary_df.to_csv(
    "development_experiment_summary.csv",
    index=False
)

selected_experiment = (
    experiment_summary_df[
        experiment_summary_df["selection_eligible"] == "Yes"
    ]
    .sort_values(
        by=[
            "overall_oof_accuracy",
            "overall_oof_spearman"
        ],
        ascending=False
    )
    .iloc[0]["experiment"]
)

print("Selected final experiment:", selected_experiment)

assert selected_experiment == "E3_NLI_similarity"